# AirQ-CV: Dual-Clock Architecture & DC-Net GPU Training
### Automated Run Pipeline (Option B: Cloud GDrive Streaming -> GPU Cache -> Training -> Empirical Evaluation)
- **Target Pollutants**: NO2, CO, SO2 (`--poll all`) or NO2 only (`--poll NO2`)
- **Historical Shortcut Prevention**: 50% Spatial Dropout on `long_mem`
- **Spatial Prior**: Per-sample batch slicing across harmonic lookup tables
- **Temporal Split**: Stage 1 (2019–2023 Train, 2024 Held-Out Validation)
- **Benchmark Metric**: Physical empirical training pixel climatology (external yardstick)


In [ ]:
import os, sys, torch
print('=== GPU ENVIRONMENT CHECK ===')
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Count: {torch.cuda.device_count()}')
    for d in range(torch.cuda.device_count()):
        print(f'  GPU {d}: {torch.cuda.get_device_name(d)} ({torch.cuda.get_device_properties(d).total_memory / (1024**3):.2f} GB)')

!pip install -q rasterio gdown lightgbm gradio


In [ ]:
import os
print('=== DEPLOYING UPDATED DUAL-CLOCK CODE MODULES ===')
with open('align_tn.py', 'w') as f:
    f.write('"""Strict S2<->S5P alignment: S2 is area-averaged onto an exact high-res subgrid per S5P cell (valid-pixel weighted)."""\nimport numpy as np, rasterio\nfrom rasterio.warp import reproject, Resampling\nfrom rasterio.transform import Affine\n\nR = 8    # Native 1:1 S2 sub-pixels per S5P cell (~600m matching raw GEE statewide export; no artificial upsampling)\n\ndef s5p_grid(s5p_path):\n    with rasterio.open(s5p_path) as src:\n        return src.transform, src.height, src.width, src.crs\n\ndef align_s2_to_s5p(s2_path, s5p_transform, H5, W5, crs, r=R):\n    """returns S2 (12, r*H5, r*W5) float32 and valid (1, r*H5, r*W5) in [0,1] = fraction of valid sub-pixels."""\n    dst_tf = s5p_transform * Affine.scale(1.0 / r, 1.0 / r)           # exact sub-grid of the S5P grid\n    with rasterio.open(s2_path) as src:\n        x = src.read().astype(np.float32)                              # (12,h,w)\n        valid = (np.nan_to_num(x[:9]) != 0).any(0).astype(np.float32)  # all 9 reflectance bands == 0  -> NoData/ocean fill\n        x = np.nan_to_num(x, nan=0.0) * valid[None]\n        out = np.zeros((x.shape[0], H5 * r, W5 * r), np.float32); vout = np.zeros((1, H5 * r, W5 * r), np.float32)\n        for b in range(x.shape[0]):\n            reproject(x[b], out[b], src_transform=src.transform, src_crs=src.crs, dst_transform=dst_tf,\n                      dst_crs=crs, resampling=Resampling.average)\n        reproject(valid, vout[0], src_transform=src.transform, src_crs=src.crs, dst_transform=dst_tf,\n                  dst_crs=crs, resampling=Resampling.average)\n    out = out / np.clip(vout, 1e-6, None)                              # valid-weighted mean\n    out[:, vout[0] < 0.05] = 0.0\n    return out, vout\n')
print('Deployed: align_tn.py')
with open('dataset.py', 'w') as f:
    f.write('import json, os, numpy as np, pandas as pd, torch\nfrom torch.utils.data import Dataset\n\nclass S5PNorm:\n    def fit(s, val, mask, idx):\n        s.s, s.mu, s.sd = [], [], []\n        for c in range(val.shape[1]):\n            x = val[idx, c][mask[idx, c] > 0]\n            sc = float(np.median(np.abs(x))) + 1e-12; z = np.arcsinh(x / sc)\n            s.s.append(sc); s.mu.append(float(z.mean())); s.sd.append(float(z.std()) + 1e-6)\n        return s\n    def fwd(s, x, c): return (np.arcsinh(x / s.s[c]) - s.mu[c]) / s.sd[c]\n    def inv(s, z, c): return np.sinh(z * s.sd[c] + s.mu[c]) * s.s[c]\n    def state(s): return dict(s=s.s, mu=s.mu, sd=s.sd)\n\nclass WeatherNorm:\n    def fit(s, wx, idx):\n        sub = wx[idx]\n        s.mu = sub.mean(0).tolist()\n        s.sd = (sub.std(0) + 1e-6).tolist()\n        return s\n    def fwd(s, x):\n        return (x - np.array(s.mu, np.float32)) / np.array(s.sd, np.float32)\n    def state(s): return dict(mu=s.mu, sd=s.sd)\n\nclass DCDataset(Dataset):\n    def __init__(s, cache, split, cfg, norm=None, s2_stats=None, wx_norm=None):\n        s.c, s.cfg, s.split = cache, cfg, split\n        s.meta = json.load(open(os.path.join(cache, "meta.json")))\n        ld = lambda n, mm=None: np.load(os.path.join(cache, n + ".npy"), mmap_mode=mm)\n        s.val, s.msk, s.s5m, s.s5ok = ld("s5p_val"), ld("s5p_mask"), ld("s5p_month"), ld("s5p_month_ok")\n        s.s2m, s.s2ok = ld("s2_month", "r"), ld("s2_month_ok", "r")\n        s.ws = pd.to_datetime(s.meta["win_start"]); s.we = s.ws + pd.Timedelta(days=5)\n        s.ms, s.me = pd.to_datetime(s.meta["month_start"]), pd.to_datetime(s.meta["month_end"])\n        s.T, s.M, s.H5, s.W5 = s.meta["T"], s.meta["M"], s.meta["H5"], s.meta["W5"]\n        s.s2_ratio = int(s.meta.get("s2_ratio", s.meta["H8"] // s.meta["H5"] if s.meta.get("H5", 0) > 0 else 8))\n        lo, hi = [pd.Timestamp(x) for x in cfg["splits"][split]]\n        s.lo, s.hi = lo, hi\n        poll_idx = cfg.get("poll_idx", [0])\n        if isinstance(poll_idx, int): poll_idx = [poll_idx]\n        s.poll_idx = poll_idx\n        s.n_poll = len(poll_idx)\n        s.origins = [i for i in range(s.T) if lo <= s.ws[i] <= hi and i >= cfg["min_origin_idx"]]\n        tr_idx = [i for i in range(s.T) if s.ws[i] <= pd.Timestamp(cfg["splits"]["train"][1])]\n        s.norm = norm or S5PNorm().fit(s.val, s.msk, tr_idx)\n        s.zs5 = np.stack([s.norm.fwd(s.val[:, c], c) for c in range(s.val.shape[1])], 1).astype(np.float32) * s.msk\n        s.s2_stats = s2_stats or s._fit_s2(tr_idx)\n        z = [s.zs5[tr_idx][:, c][s.msk[tr_idx][:, c] > 0] for c in s.poll_idx]\n        s.peak_thr = np.array([np.quantile(x, 0.9) if len(x) > 0 else 1.0 for x in z], np.float32)\n        if 2 in s.poll_idx:\n            so2_pos = s.poll_idx.index(2)\n            s.so2_thr = float(np.quantile(z[so2_pos], cfg.get("so2_event_q", 0.85))) if len(z[so2_pos]) > 0 else 0.0\n        else:\n            s.so2_thr = 0.0\n\n        # Weather conditioning\n        wx_file = os.path.join(cache, "weather_5d.npy")\n        s.has_weather = os.path.exists(wx_file)\n        if s.has_weather:\n            s.wx_raw = np.load(wx_file)\n            s.wx_norm = wx_norm or WeatherNorm().fit(s.wx_raw, tr_idx)\n            s.wx_feat = s.wx_norm.fwd(s.wx_raw)\n        else:\n            s.wx_feat = None\n            s.wx_norm = None\n\n    def _fit_s2(s, tr_idx):\n        mm = [m for m in range(s.M) if s.me[m] <= pd.Timestamp(s.cfg["splits"]["train"][1])]\n        x = np.asarray(s.s2m[mm[::max(1, len(mm)//6)]], np.float32); ok = np.asarray(s.s2ok[mm[::max(1, len(mm)//6)]]) > 0\n        mu = [float(x[:, c][ok].mean()) for c in range(12)]; sd = [float(x[:, c][ok].std()) + 1e-6 for c in range(12)]\n        return dict(mu=mu, sd=sd)\n\n    def __len__(s): return s.cfg["samples_per_epoch"] if s.split == "train" else len(s.origins)\n\n    def _s2_norm(s, x):\n        return (x - np.array(s.s2_stats["mu"], np.float32)[:, None, None]) / np.array(s.s2_stats["sd"], np.float32)[:, None, None]\n\n    def _nanmed(s, months, sl):\n        if len(months) == 1:\n            m = months[0]\n            med = np.asarray(s.s2m[m][(slice(None),) + sl], np.float32)\n            ok = (np.asarray(s.s2ok[m][sl]) > 0) & np.isfinite(med).all(0)\n            return np.where(ok[None], med, 0.0).astype(np.float32), ok\n        a = np.stack([np.asarray(s.s2m[m][(slice(None),) + sl], np.float32) for m in months])\n        ok = np.stack([np.asarray(s.s2ok[m][sl]) > 0 for m in months]) & np.isfinite(a).all(1)\n        cnt = ok.sum(0)\n        # masked mean (NaN-safe: NaN*0 = NaN, so select with np.where instead of multiplying by the mask)\n        med = np.where(ok[:, None], a, 0.0).sum(0) / np.maximum(cnt[None], 1.0)\n        return med.astype(np.float32), cnt > 0\n\n    @staticmethod\n    def calc_biophysical_indices(bands):\n        """\n        Rank 1 Biophysical Index Stacking:\n        bands[0]: B1 (Coastal Aerosol)\n        bands[1]: B2 (Blue)\n        bands[3]: B4 (Red)\n        bands[4]: B5 (Red Edge 1)\n        bands[7]: B8 (Broad NIR)\n        bands[10]: B11 (SWIR 1)\n        """\n        eps = 1e-6\n        bands = np.nan_to_num(np.asarray(bands, np.float32), nan=0.0, posinf=0.0, neginf=0.0)\n        b1, b2, b4, b5, b8, b11 = bands[0], bands[1], bands[3], bands[4], bands[7], bands[10]\n        with np.errstate(divide="ignore", invalid="ignore"):\n            ndbi = np.clip((b11 - b8) / (b11 + b8 + eps), -1.0, 1.0)         # Built-up / Impervious Industrial\n            ndvi = np.clip((b8 - b4) / (b8 + b4 + eps), -1.0, 1.0)           # Vegetation Canopy Sink\n            ndre = np.clip((b8 - b5) / (b8 + b5 + eps), -1.0, 1.0)           # Red Edge Chlorophyll Sensitivity\n            aer = np.clip(b1 / (b2 + eps), 0.0, 5.0) / 5.0                   # Aerosol Optical Ratio in [0, 1]\n        return np.nan_to_num(np.stack([ndbi, ndvi, ndre, aer], axis=0), nan=0.0)\n\n    def slow(s, t_end, sl, hw_res):\n        lag = pd.Timedelta(days=s.cfg["s2_lag_days"])\n        use = [m for m in range(s.M) if s.me[m] + lag <= t_end][-3:]\n        # 32 channels: 12 bands + 4 biophysical indices + 12 diff bands + 2 diff indices + 2 masks\n        out = np.zeros((32,) + hw_res, np.float32)\n        if not use: return out\n        now, ok = s._nanmed(use, sl)\n        now_norm = np.nan_to_num(s._s2_norm(now), nan=0.0) * ok\n        out[:12] = now_norm\n        idx_now = s.calc_biophysical_indices(now) * ok\n        out[12:16] = idx_now\n        out[30] = ok\n        prev = [m - 12 for m in use]\n        if min(prev) >= 0:\n            pv, pok = s._nanmed(prev, sl); both = ok & pok\n            pv_norm = np.nan_to_num(s._s2_norm(pv), nan=0.0)\n            out[16:28] = (now_norm - pv_norm) * both\n            idx_pv = s.calc_biophysical_indices(pv)\n            out[28] = (idx_now[0] - idx_pv[0]) * both                    # Delta NDBI\n            out[29] = (idx_now[1] - idx_pv[1]) * both                    # Delta NDVI\n            out[31] = 1.0\n        return out\n\n    def __getitem__(s, k):\n        cfg = s.cfg; rng = np.random\n        o = s.origins[rng.randint(len(s.origins))] if s.split == "train" else s.origins[k]\n        P = min(cfg["patch"], s.H5, s.W5)\n        if s.split == "train":\n            i0 = rng.randint(0, max(1, s.H5 - P + 1)) if s.H5 > P else 0\n            j0 = rng.randint(0, max(1, s.W5 - P + 1)) if s.W5 > P else 0\n            h = w = P\n        else: i0 = j0 = 0; h, w = s.H5, s.W5\n        Hp, Wp = (-(-h // 8)) * 8, (-(-w // 8)) * 8\n        pad = lambda a, k=1: np.pad(a, [(0, 0)] * (a.ndim - 2) + [(0, Hp * k - a.shape[-2]), (0, Wp * k - a.shape[-1])])\n        L = cfg["L"]; ids = np.arange(o - L + 1, o + 1); okid = ids >= 0; ids = np.clip(ids, 0, None)\n        fv = s.zs5[ids][:, s.poll_idx, i0:i0 + h, j0:j0 + w]\n        fm = s.msk[ids][:, s.poll_idx, i0:i0 + h, j0:j0 + w].astype(np.float32) * okid[:, None, None, None]\n        if s.split == "train" and cfg.get("mask_aug", 0.0) > 0:\n            drop = (rng.rand(*fm.shape) < cfg["mask_aug"]); fm = fm * (~drop)\n        fv = fv * fm\n        t_end = s.we[o]\n        use = [m for m in range(s.M) if s.me[m] <= t_end][-12:]\n        lv = np.zeros((12, s.n_poll, h, w), np.float32); lk = np.zeros((12, s.n_poll, h, w), np.float32)\n        for q, m in enumerate(use[::-1]):\n            for p_i, c in enumerate(s.poll_idx):\n                mk = s.s5ok[m][c, i0:i0 + h, j0:j0 + w].astype(np.float32)\n                lv[q, p_i] = s.norm.fwd(s.s5m[m][c, i0:i0 + h, j0:j0 + w], c) * mk\n                lk[q, p_i] = mk\n        long_mem = np.concatenate([lv.reshape(12 * s.n_poll, h, w), lk.reshape(12 * s.n_poll, h, w)])\n\n        # Spatial / Historical Dropout on long_mem to prevent S2 shortcut\n        if s.split == "train" and cfg.get("long_mem_dropout", 0.0) > 0:\n            if rng.rand() < cfg["long_mem_dropout"]:\n                long_mem = np.zeros_like(long_mem)\n\n        r = s.s2_ratio\n        sl = (slice(i0 * r, (i0 + h) * r), slice(j0 * r, (j0 + w) * r))\n        s2 = s.slow(t_end, sl, (h * r, w * r))\n        steps = cfg["steps"]; y = np.zeros((len(steps), s.n_poll, h, w), np.float32); ym = np.zeros_like(y)\n        for q, st in enumerate(steps):\n            t = o + st\n            if t < s.T and s.lo <= s.ws[t] <= s.hi:\n                for p_i, c in enumerate(s.poll_idx):\n                    y[q, p_i] = s.zs5[t][c, i0:i0 + h, j0:j0 + w]\n                    ym[q, p_i] = s.msk[t][c, i0:i0 + h, j0:j0 + w]\n        if 2 in s.poll_idx:\n            so2_pos = s.poll_idx.index(2)\n            ev = ((y[:, so2_pos] > s.so2_thr) & (ym[:, so2_pos] > 0)).astype(np.float32)\n        else:\n            ev = np.zeros((len(steps), h, w), np.float32)\n        doy = lambda ts: ts.dayofyear + ts.hour / 24.0\n        d_org = doy(s.we[o]); d_tgt = np.array([(s.we[o] + pd.Timedelta(days=5 * st)).dayofyear for st in steps], np.float32)\n        b = dict(fast_v=pad(fv), fast_m=pad(fm), s2_slow=pad(s2, r), long_mem=pad(long_mem), y=pad(y), ym=pad(ym), ev=pad(ev),\n                 doy_tgt=d_tgt, doy_org_sc=np.array([np.sin(2 * np.pi * d_org / 365.25), np.cos(2 * np.pi * d_org / 365.25)], np.float32),\n                 crop=np.array([i0, j0, h, w], dtype=np.int64))\n        if s.has_weather and s.wx_feat is not None:\n            b["weather"] = s.wx_feat[o].astype(np.float32)\n        for k in ("fast_v", "fast_m", "s2_slow", "long_mem", "y", "ym"):\n            if not np.isfinite(b[k]).all():\n                raise ValueError(f"non-finite values in \'{k}\' (origin={o}, crop={i0},{j0}) - check the cache")\n        return {k: torch.from_numpy(np.asarray(v)) for k, v in b.items()}\n\ndef collate(b):\n    return {k: torch.stack([x[k] for x in b]) for k in b[0]}\n')
print('Deployed: dataset.py')
with open('dcnet.py', 'w') as f:
    f.write('"""DC-Net: Dual-Clock S2+S5P multi-horizon forecaster (pure deep learning)."""\nimport math, torch, torch.nn as nn, torch.nn.functional as F\n\nHORIZONS_DAYS = (5, 10, 20, 30, 60, 120, 150)      # 5-day steps: 1,2,4,6,12,24,30\nHORIZON_STEPS = tuple(d // 5 for d in HORIZONS_DAYS)\n\ndef gn(c, g=8): return nn.GroupNorm(min(g, c), c)\n\nclass ResBlock(nn.Module):\n    def __init__(s, cin, cout, p_drop=0.0):\n        super().__init__()\n        s.n1, s.c1 = gn(cin), nn.Conv2d(cin, cout, 3, padding=1, padding_mode="reflect")\n        s.n2, s.c2 = gn(cout), nn.Conv2d(cout, cout, 3, padding=1, padding_mode="reflect")\n        s.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()\n        s.drop = nn.Dropout2d(p_drop)\n    def forward(s, x):\n        h = s.c1(F.silu(s.n1(x))); h = s.c2(s.drop(F.silu(s.n2(h))))\n        return h + s.skip(x)\n\nclass ConvGRUCell(nn.Module):\n    def __init__(s, cin, ch, k=3):\n        super().__init__()\n        s.ch = ch\n        s.gates = nn.Conv2d(cin + ch, 2 * ch, k, padding=k // 2, padding_mode="reflect")\n        s.cand = nn.Conv2d(cin + ch, ch, k, padding=k // 2, padding_mode="reflect")\n    def forward(s, x, h):\n        if h is None: h = x.new_zeros(x.size(0), s.ch, *x.shape[-2:])\n        z, r = torch.sigmoid(s.gates(torch.cat([x, h], 1))).chunk(2, 1)\n        n = torch.tanh(s.cand(torch.cat([x, r * h], 1)))\n        return (1 - z) * h + z * n\n\nclass PartialStem(nn.Module):\n    """Mask-aware first conv (partial-conv style): invalid S5P pixels never leak as zeros."""\n    def __init__(s, c_val, cout):\n        super().__init__()\n        s.conv = nn.Conv2d(c_val, cout, 3, padding=1, padding_mode="reflect", bias=False)\n        s.bias = nn.Parameter(torch.zeros(1, cout, 1, 1))\n        s.register_buffer("ones", torch.ones(1, 1, 3, 3))\n        s.mix = nn.Conv2d(cout + c_val, cout, 1)\n    def forward(s, v, m):\n        vm = v * m\n        y = s.conv(vm)\n        msum = F.conv2d(m.mean(1, keepdim=True), s.ones, padding=1)\n        y = y * (9.0 / msum.clamp(min=1.0)) * (msum > 0) + s.bias\n        return F.silu(s.mix(torch.cat([y, m], 1)))\n\nclass FiLM(nn.Module):\n    def __init__(s, cond, ch):\n        super().__init__(); s.fc = nn.Linear(cond, 2 * ch); nn.init.zeros_(s.fc.weight); nn.init.zeros_(s.fc.bias)\n    def forward(s, x, c):\n        g, b = s.fc(c).chunk(2, 1)\n        return x * (1 + g[..., None, None]) + b[..., None, None]\n\nclass RelPosAttnBlock(nn.Module):\n    def __init__(s, dim, heads=6, p=0.1):\n        super().__init__()\n        s.n1, s.n2 = nn.LayerNorm(dim), nn.LayerNorm(dim)\n        s.attn = nn.MultiheadAttention(dim, heads, dropout=p, batch_first=True)\n        s.mlp = nn.Sequential(nn.Linear(dim, 2 * dim), nn.GELU(), nn.Dropout(p), nn.Linear(2 * dim, dim))\n        s.dw = nn.Conv2d(dim, dim, 3, padding=1, groups=dim, padding_mode="reflect")\n    def forward(s, x):\n        B, C, H, W = x.shape\n        x = x + s.dw(x)\n        t = x.flatten(2).transpose(1, 2)\n        h = s.n1(t); t = t + s.attn(h, h, h, need_weights=False)[0]\n        t = t + s.mlp(s.n2(t))\n        return t.transpose(1, 2).reshape(B, C, H, W)\n\nclass FastEncoder(nn.Module):\n    def __init__(s, c_in=3, ch=(32, 64, 128, 192), gru=(64, 128, 128), p=0.05):\n        super().__init__()\n        s.stem = PartialStem(c_in, ch[0])\n        s.e1 = ResBlock(ch[0], ch[0], p)\n        s.d1, s.e2 = nn.Conv2d(ch[0], ch[1], 3, 2, 1), ResBlock(ch[1], ch[1], p)\n        s.d2, s.e3 = nn.Conv2d(ch[1], ch[2], 3, 2, 1), ResBlock(ch[2], ch[2], p)\n        s.d3, s.e4 = nn.Conv2d(ch[2], ch[3], 3, 2, 1), ResBlock(ch[3], ch[3], p)\n        s.g2, s.g3, s.g4 = ConvGRUCell(ch[1], gru[0]), ConvGRUCell(ch[2], gru[1]), ConvGRUCell(ch[3], gru[2])\n        s.last1 = nn.Conv2d(ch[0], ch[0], 1)\n    def forward(s, v, m):\n        B, L = v.shape[:2]\n        h2 = h3 = h4 = None\n        for t in range(L):\n            x1 = s.e1(s.stem(v[:, t], m[:, t]))\n            x2 = s.e2(s.d1(x1)); x3 = s.e3(s.d2(x2)); x4 = s.e4(s.d3(x3))\n            h2, h3, h4 = s.g2(x2, h2), s.g3(x3, h3), s.g4(x4, h4)\n        return s.last1(x1), h2, h3, h4\n\nclass S2SlowEncoder(nn.Module):\n    def __init__(s, c_in=32, ch=(24, 32, 48, 64), p=0.05):\n        super().__init__()\n        s.stem = nn.Conv2d(c_in, ch[0], 3, padding=1, padding_mode="reflect")\n        s.b1 = ResBlock(ch[0], ch[0], p)\n        s.d1, s.b2 = nn.Conv2d(ch[0], ch[1], 3, 2, 1), ResBlock(ch[1], ch[1], p)\n        s.d2, s.b4 = nn.Conv2d(ch[1], ch[2], 3, 2, 1), ResBlock(ch[2], ch[2], p)\n        s.d3, s.b8 = nn.Conv2d(ch[2], ch[3], 3, 2, 1), ResBlock(ch[3], ch[3], p)\n    def forward(s, x):\n        # Adaptive handling for both 32 biophysical stacked channels and 26 legacy channels\n        if x.size(1) != s.stem.in_channels:\n            if x.size(1) < s.stem.in_channels:\n                x = F.pad(x, (0, 0, 0, 0, 0, s.stem.in_channels - x.size(1)))\n            else:\n                x = x[:, :s.stem.in_channels]\n        f1 = s.b1(F.silu(s.stem(x)))\n        f2 = s.b2(s.d1(f1)); f4 = s.b4(s.d2(f2)); f8 = s.b8(s.d3(f4))\n        return f1, f2, f4, f8\n\nclass FineDisaggregationHead(nn.Module):\n    """\n    Rank 1 Physics-Guided Fine Disaggregation Head:\n    Fuses high-res S2 biophysical features (NDBI/NDVI/SWIR) with coarse regional latent state\n    to predict fine-scale emission modulation at native spatial resolution.\n    """\n    def __init__(s, c_trunk=64, c_f1=24, n_poll=1, p=0.05):\n        super().__init__()\n        s.fuse = nn.Sequential(\n            nn.Conv2d(c_trunk + c_f1, 48, 3, padding=1, padding_mode="reflect"),\n            nn.SiLU(),\n            ResBlock(48, 48, p),\n            nn.Conv2d(48, n_poll, 1)\n        )\n        nn.init.zeros_(s.fuse[-1].weight)\n        nn.init.zeros_(s.fuse[-1].bias)\n    def forward(s, trunk, f1):\n        trunk_up = F.interpolate(trunk, size=f1.shape[-2:], mode="bilinear", align_corners=False)\n        return s.fuse(torch.cat([trunk_up, f1], 1))\n\nclass LongMemoryEncoder(nn.Module):\n    def __init__(s, c_in=72, ch=64, p=0.05):\n        super().__init__()\n        s.net = nn.Sequential(nn.Conv2d(c_in, ch, 3, padding=1, padding_mode="reflect"), nn.SiLU(),\n                              ResBlock(ch, ch, p), ResBlock(ch, ch, p))\n    def forward(s, x): return s.net(x)\n\nclass AttnGate(nn.Module):\n    def __init__(s, cx, cg, ci):\n        super().__init__()\n        s.wx, s.wg, s.psi = nn.Conv2d(cx, ci, 1), nn.Conv2d(cg, ci, 1), nn.Conv2d(ci, 1, 1)\n    def forward(s, x, g):\n        return x * torch.sigmoid(s.psi(F.silu(s.wx(x) + s.wg(g))))\n\nclass SeasonalPrior(nn.Module):\n    def __init__(s, n_ch, H, W, K=3):\n        super().__init__()\n        s.K = K\n        s.W = nn.Parameter(torch.zeros(n_ch, 1 + 2 * K, H, W))\n        s.smooth = nn.Conv2d(n_ch * (1 + 2 * K), n_ch * (1 + 2 * K), 5, padding=2, groups=n_ch * (1 + 2 * K), bias=False)\n        with torch.no_grad(): s.smooth.weight.fill_(1 / 25.)\n    def basis(s, doy):\n        k = torch.arange(1, s.K + 1, device=doy.device).float()\n        ang = 2 * math.pi * doy[..., None] / 365.25 * k\n        return torch.cat([torch.ones_like(doy)[..., None], ang.sin(), ang.cos()], -1)\n    def forward(s, doy, crop, out_hw):\n        Wm = s.W.flatten(0, 1)[None]\n        Wm = s.smooth(Wm)[0].view_as(s.W)\n        basis = s.basis(doy)\n        B = doy.size(0)\n        if torch.is_tensor(crop) and crop.dim() == 2 and crop.size(0) == B:\n            slices = []\n            for b in range(B):\n                i0, j0, h, w = int(crop[b, 0]), int(crop[b, 1]), int(crop[b, 2]), int(crop[b, 3])\n                sub = Wm[:, :, i0:i0 + h, j0:j0 + w]\n                ph, pw = out_hw[0] - sub.shape[-2], out_hw[1] - sub.shape[-1]\n                if ph > 0 or pw > 0: sub = F.pad(sub, (0, max(pw, 0), 0, max(ph, 0)), mode="replicate")\n                slices.append(sub)\n            Wm_b = torch.stack(slices, 0)\n            return torch.einsum("bqk,bckhw->bqchw", basis, Wm_b)\n        else:\n            if torch.is_tensor(crop) and crop.dim() == 2:\n                i0, j0, h, w = int(crop[0, 0]), int(crop[0, 1]), int(crop[0, 2]), int(crop[0, 3])\n            elif torch.is_tensor(crop) and crop.dim() == 1:\n                i0, j0, h, w = int(crop[0]), int(crop[1]), int(crop[2]), int(crop[3])\n            else:\n                i0, j0, h, w = [int(v) for v in crop]\n            sub = Wm[:, :, i0:i0 + h, j0:j0 + w]\n            ph, pw = out_hw[0] - sub.shape[-2], out_hw[1] - sub.shape[-1]\n            if ph > 0 or pw > 0: sub = F.pad(sub, (0, max(pw, 0), 0, max(ph, 0)), mode="replicate")\n            return torch.einsum("bqk,ckhw->bqchw", basis, sub)\n\nclass MonotoneGate(nn.Module):\n    def __init__(s, n_ch):\n        super().__init__(); s.raw = nn.Parameter(torch.full((n_ch,), -0.5))\n    def forward(s, h_days):\n        return torch.exp(-F.softplus(s.raw)[None] * (h_days[:, None] / 150.0))\n\nclass DCNet(nn.Module):\n    def __init__(s, n_poll=1, full_hw=(123, 95), horizons=HORIZONS_DAYS, fine_ratio=8,\n                 dim=192, attn_layers=2, heads=6, p=0.1, use_B=True, use_s2=True, use_weather=True, so2_idx=2,\n                 long_mem_dropout=0.0):\n        super().__init__()\n        s.n_poll, s.horizons, s.use_B, s.use_s2, s.use_weather, s.so2_idx = n_poll, horizons, use_B, use_s2, use_weather, so2_idx\n        s.long_mem_dropout = long_mem_dropout\n        s.Q = len(horizons)\n        s.register_buffer("h_days", torch.tensor(horizons, dtype=torch.float32))\n        s.fast = FastEncoder(c_in=n_poll)\n        s.slow = S2SlowEncoder(c_in=32)\n        s.fine_head = FineDisaggregationHead(c_trunk=64, c_f1=24, n_poll=n_poll, p=p)\n        s.long = LongMemoryEncoder(c_in=12 * n_poll * 2)\n        n_cond = 2 + 2 * s.Q + (8 if use_weather else 0)\n        cond = 32\n        s.cal = nn.Sequential(nn.Linear(n_cond, cond), nn.SiLU(), nn.Linear(cond, cond))\n        s.fuse = nn.Conv2d(128 + 64 + 64, dim, 1)\n        s.film_b = FiLM(cond, dim)\n        s.attn = nn.ModuleList([RelPosAttnBlock(dim, heads, p) for _ in range(attn_layers)])\n        s.up3, s.ag3, s.dec3 = nn.Conv2d(dim, 128, 1), AttnGate(128, 64, 64), ResBlock(128 + 128, 128, 0.05)\n        s.up2, s.ag2, s.dec2 = nn.Conv2d(128, 64, 1), AttnGate(64, 64, 32), ResBlock(64 + 64, 64, 0.05)\n        s.up1, s.ag1, s.dec1 = nn.Conv2d(64, 32, 1), AttnGate(32, 64, 32), ResBlock(32 + 32, 32, 0.05)\n        s.film_d = FiLM(cond, 32)\n        s.mix_native = nn.Conv2d(32 + 64 + 64, 64, 3, padding=1, padding_mode="reflect")\n        s.head = nn.Conv2d(64, s.Q * n_poll * 3, 1)\n        s.occ = nn.Conv2d(64, s.Q, 1)\n        s.prior = SeasonalPrior(n_poll, *full_hw)\n        s.gate = MonotoneGate(n_poll)\n        s.Bhead = nn.Sequential(nn.Conv2d(64 + 64, 32, 3, padding=1, padding_mode="reflect"), nn.SiLU(),\n                                nn.Conv2d(32, n_poll, 1))\n        nn.init.zeros_(s.Bhead[-1].weight); nn.init.zeros_(s.Bhead[-1].bias)\n        nn.init.zeros_(s.head.bias)\n\n    def forward(s, batch):\n        v, m = batch["fast_v"], batch["fast_m"]\n        s2, lm = batch["s2_slow"], batch["long_mem"]\n        crop, doy_tgt = batch["crop"], batch["doy_tgt"]\n        B = v.size(0)\n\n        # Spatial / Historical Dropout on long_mem to prevent S2 shortcut\n        if s.training and s.long_mem_dropout > 0:\n            drop_mask = (torch.rand(B, 1, 1, 1, device=lm.device) >= s.long_mem_dropout).float()\n            lm = lm * drop_mask\n\n        cal_items = [batch["doy_org_sc"], torch.sin(2*math.pi*doy_tgt/365.25), torch.cos(2*math.pi*doy_tgt/365.25)]\n        if s.use_weather and "weather" in batch and batch["weather"] is not None:\n            cal_items.append(batch["weather"])\n        cal = s.cal(torch.cat(cal_items, 1))\n\n        f1, h2, h3, h4 = s.fast(v, m)\n        if s.use_s2: s1, s2f, s4f, s8f = s.slow(s2)\n        else:\n            H, W = v.shape[-2:]\n            s8f = v.new_zeros(B, 64, H, W); s4f = v.new_zeros(B, 48, 2*H, 2*W); s2f = v.new_zeros(B, 32, 4*H, 4*W)\n            s1 = v.new_zeros(B, 24, H, W)\n        lmf = s.long(lm)\n        s8_native = F.interpolate(s8f, size=f1.shape[-2:], mode="bilinear", align_corners=False)\n        s8_h2 = F.interpolate(s8f, size=h2.shape[-2:], mode="bilinear", align_corners=False)\n        s8_h3 = F.interpolate(s8f, size=h3.shape[-2:], mode="bilinear", align_corners=False)\n        z = s.fuse(torch.cat([h4, F.adaptive_avg_pool2d(s8f, h4.shape[-2:]), F.adaptive_avg_pool2d(lmf, h4.shape[-2:])], 1))\n        z = s.film_b(z, cal)\n        for blk in s.attn: z = blk(z)\n        d3 = s.up3(F.interpolate(z, size=h3.shape[-2:], mode="bilinear", align_corners=False))\n        d3 = s.dec3(torch.cat([d3, s.ag3(h3, s8_h3)], 1))\n        d2 = s.up2(F.interpolate(d3, size=h2.shape[-2:], mode="bilinear", align_corners=False))\n        d2 = s.dec2(torch.cat([d2, s.ag2(h2, s8_h2)], 1))\n        d1 = s.up1(F.interpolate(d2, size=f1.shape[-2:], mode="bilinear", align_corners=False))\n        d1 = s.dec1(torch.cat([d1, s.ag1(f1, s8_native)], 1))\n        d1 = s.film_d(d1, cal)\n        trunk = F.silu(s.mix_native(torch.cat([d1, s8_native, lmf], 1)))\n        out = s.head(trunk).view(B, s.Q, s.n_poll, 3, *trunk.shape[-2:])\n        a, wlo, whi = out[:, :, :, 0], F.softplus(out[:, :, :, 1]) + 1e-3, F.softplus(out[:, :, :, 2]) + 1e-3\n        prior = s.prior(doy_tgt, crop, trunk.shape[-2:])\n        g = s.gate(s.h_days)[None, :, :, None, None]\n        Bterm = s.Bhead(torch.cat([trunk, s8_native], 1))[:, None] if (s.use_B and s.use_s2) else 0.0\n        bscale = (1 - g)\n        med = prior + bscale * Bterm + g * a\n\n        # Rank 1: High-Resolution Disaggregation & Fine-Scale Prediction\n        fine_mod = s.fine_head(trunk, s1) if s.use_s2 else torch.zeros(B, s.n_poll, *s1.shape[-2:], device=med.device)\n        med_up = F.interpolate(med.flatten(0, 1), size=s1.shape[-2:], mode="bilinear", align_corners=False).view(B, s.Q, s.n_poll, *s1.shape[-2:])\n        fine_q50 = med_up + fine_mod[:, None]\n\n        return {"q50": med, "q10": med - wlo, "q90": med + whi, "fine_q50": fine_q50, "occ_logit": s.occ(trunk), "B": Bterm}\n\ndef masked_pinball(pred, target, mask, tau):\n    e = target - pred\n    l = torch.maximum(tau * e, (tau - 1) * e) * mask\n    return l.sum() / mask.sum().clamp(min=1.0)\n\ndef dcnet_loss(out, y, ymask, so2_event=None, so2_idx=2, lam_grad=0.1, lam_peak=0.5, lam_B=1e-3, lam_occ=0.2,\n               lam_mass=0.1, peak_thr=None, hweights=None):\n    tot = 0.0\n    q50, q10, q90 = out["q50"], out["q10"], out["q90"]\n    n_poll = y.size(2)\n    w = 1.0\n    if peak_thr is not None:\n        peak_t = torch.as_tensor(peak_thr, dtype=y.dtype, device=y.device)\n        if peak_t.numel() == n_poll:\n            w = 1.0 + lam_peak * torch.sigmoid((y - peak_t[None, None, :, None, None]) / 0.25)\n    m = ymask * w\n    tot = tot + masked_pinball(q50, y, m, 0.5) + 0.5 * masked_pinball(q10, y, m, 0.1) + 0.5 * masked_pinball(q90, y, m, 0.9)\n    gx = lambda t: t[..., :, 1:] - t[..., :, :-1]\n    gy = lambda t: t[..., 1:, :] - t[..., :-1, :]\n    mx, my = ymask[..., :, 1:] * ymask[..., :, :-1], ymask[..., 1:, :] * ymask[..., :-1, :]\n    tot = tot + lam_grad * ((gx(q50) - gx(y)).abs() * mx).sum() / mx.sum().clamp(min=1) \\\n                + lam_grad * ((gy(q50) - gy(y)).abs() * my).sum() / my.sum().clamp(min=1)\n\n    # Rank 1: Area-to-Point Mass Conservation Integral Loss (Late Pooling)\n    if "fine_q50" in out and out["fine_q50"] is not None:\n        fine_pred = out["fine_q50"]\n        coarse_pool = F.adaptive_avg_pool2d(fine_pred.flatten(0, 1), y.shape[-2:]).view_as(y)\n        mass_loss = ((coarse_pool - y).abs() * m).sum() / m.sum().clamp(min=1.0)\n        tot = tot + lam_mass * mass_loss\n\n    if so2_event is not None and so2_idx is not None and so2_idx < n_poll:\n        mk = ymask[:, :, so2_idx]\n        bce = F.binary_cross_entropy_with_logits(out["occ_logit"], so2_event, reduction="none") * mk\n        tot = tot + lam_occ * bce.sum() / mk.sum().clamp(min=1)\n    if torch.is_tensor(out["B"]): tot = tot + lam_B * out["B"].abs().mean()\n    return tot\n\ndef count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)\n')
print('Deployed: dcnet.py')
with open('train.py', 'w') as f:
    f.write('import argparse, copy, json, math, os, time, numpy as np, torch\nfrom torch.utils.data import DataLoader\nfrom dataset import DCDataset, collate\nfrom dcnet import DCNet, dcnet_loss, count_params, HORIZON_STEPS\n\nSPLITS_STAGE1 = dict(train=["2019-01-01", "2023-12-31"], val=["2024-01-01", "2024-12-31"], test=["2025-01-01", "2025-12-31"])\nSPLITS_STAGE2 = dict(train=["2019-01-01", "2024-12-31"], val=["2025-01-01", "2025-12-31"], test=["2026-01-01", "2026-12-31"])\nSPLITS_LEGACY = dict(train=["2019-01-01", "2022-12-31"], val=["2023-01-01", "2023-12-31"], test=["2024-01-01", "2024-12-31"])\n\nCFG = dict(L=12, steps=list(HORIZON_STEPS), patch=64, s2_lag_days=20, so2_event_q=0.85, mask_aug=0.1, min_origin_idx=12,\n           samples_per_epoch=400, batch=8, epochs=120, lr=3e-4, wd=0.05, warmup=3, ema=0.999, clip=1.0, patience=15,\n           n_poll=1, poll_idx=[0], long_mem_dropout=0.5,\n           splits=SPLITS_STAGE1)\n\ndef to_dev(b, dev): return {k: (v.to(dev, non_blocking=True) if torch.is_tensor(v) else v) for k, v in b.items()}\ndef model_in(b):\n    res = dict(fast_v=b["fast_v"], fast_m=b["fast_m"], s2_slow=b["s2_slow"], long_mem=b["long_mem"],\n               crop=b["crop"], doy_tgt=b["doy_tgt"], doy_org_sc=b["doy_org_sc"])\n    if "weather" in b: res["weather"] = b["weather"]\n    return res\n\n@torch.no_grad()\ndef evaluate(model, dl, dev, peak, n_poll=1):\n    model.eval(); tot, n = 0.0, 0\n    cov = np.zeros((7, n_poll)); cn = np.zeros((7, n_poll)); sse = np.zeros((7, n_poll)); sst = np.zeros((7, n_poll))\n    for b in dl:\n        b = to_dev(b, dev)\n        if torch.is_tensor(b["crop"]):\n            if b["crop"].dim() == 2:\n                H, W = int(b["crop"][0, 2]), int(b["crop"][0, 3])\n            else:\n                H, W = int(b["crop"][2]), int(b["crop"][3])\n        else:\n            H, W = int(b["crop"][2]), int(b["crop"][3])\n\n        o = model(model_in(b)); crop_coarse = lambda t: t[..., :H, :W]\n        y, ym = crop_coarse(b["y"]), crop_coarse(b["ym"])\n        out = {}\n        for k, v in o.items():\n            if torch.is_tensor(v) and v.dim() >= 4:\n                if k.startswith("fine_"):\n                    r = max(1, v.shape[-1] // max(1, o["q50"].shape[-1]))\n                    out[k] = v[..., :H * r, :W * r]\n                else:\n                    out[k] = crop_coarse(v)\n            else:\n                out[k] = v\n        tot += dcnet_loss(out, y, ym, crop_coarse(b["ev"]), peak_thr=peak).item(); n += 1\n        inside = ((y >= out["q10"]) & (y <= out["q90"])).float() * ym\n        cov += inside.sum((0, 3, 4)).cpu().numpy(); cn += ym.sum((0, 3, 4)).cpu().numpy()\n        e = ((y - out["q50"]) ** 2 * ym).sum((0, 3, 4)).cpu().numpy(); sse += e\n        mu = (y * ym).sum((0, 3, 4)) / ym.sum((0, 3, 4)).clamp(min=1); sst += (((y - mu[None, :, :, None, None]) ** 2) * ym).sum((0, 3, 4)).cpu().numpy()\n    r2 = np.where(sst > 1e-9, 1 - sse / np.maximum(sst, 1e-9), np.nan)\n    return tot / max(n, 1), r2, cov / np.maximum(cn, 1)\n\ndef main(a):\n    cfg = copy.deepcopy(CFG)\n\n    # Stage split selection\n    if a.stage == "1":\n        cfg["splits"] = SPLITS_STAGE1\n    elif a.stage == "2":\n        cfg["splits"] = SPLITS_STAGE2\n    elif a.stage == "legacy":\n        cfg["splits"] = SPLITS_LEGACY\n\n    # Pollutant selection (Single NO2 vs multi-pollutant)\n    if a.poll == "all" or a.n_poll == 3:\n        cfg["n_poll"] = 3\n        cfg["poll_idx"] = [0, 1, 2]\n    elif a.poll == "NO2" or a.n_poll == 1:\n        cfg["n_poll"] = 1\n        cfg["poll_idx"] = [0]\n\n    if a.long_mem_dropout is not None:\n        cfg["long_mem_dropout"] = a.long_mem_dropout\n\n    # Custom cfg overrides any defaults\n    if a.cfg: cfg.update(json.load(open(a.cfg)))\n    if a.batch: cfg["batch"] = a.batch\n    if a.max_epochs and a.max_epochs > 0: cfg["epochs"] = a.max_epochs\n\n    torch.manual_seed(a.seed); np.random.seed(a.seed)\n\n    # Hardware Safety Rails (Multi-GPU & Laptop Protection)\n    num_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0\n    if torch.cuda.is_available():\n        for d_idx in range(num_gpus):\n            torch.cuda.set_per_process_memory_fraction(a.vram_cap, device=d_idx)\n        torch.cuda.empty_cache()\n        print(f"[Hardware Guard] {num_gpus} GPU(s) detected. Per-device VRAM fraction capped at {a.vram_cap*100:.0f}%.")\n\n    dev = "cuda" if torch.cuda.is_available() else "cpu"\n    tr = DCDataset(a.cache, "train", cfg)\n    va = DCDataset(a.cache, "val", cfg, tr.norm, tr.s2_stats, tr.wx_norm)\n    tl = DataLoader(tr, cfg["batch"], num_workers=a.workers, collate_fn=collate, drop_last=True, persistent_workers=(a.workers > 0))\n    vl = DataLoader(va, 1, num_workers=0, collate_fn=collate)\n    raw_model = DCNet(tr.n_poll, (tr.H5, tr.W5), use_B=bool(a.use_B), use_s2=bool(a.use_s2),\n                      use_weather=bool(a.use_weather), long_mem_dropout=cfg.get("long_mem_dropout", 0.0)).to(dev)\n    use_dp = (dev == "cuda" and num_gpus > 1 and not getattr(a, "no_dp", False))\n    if use_dp:\n        print(f"[Multi-GPU] Initializing torch.nn.DataParallel across {num_gpus} GPUs: {[torch.cuda.get_device_name(i) for i in range(num_gpus)]}")\n        print(f"            Global batch: {cfg[\'batch\']} ({cfg[\'batch\'] // num_gpus} per GPU).")\n        model = torch.nn.DataParallel(raw_model)\n    else:\n        model = raw_model\n\n    print(f"DC-Net initialized on {dev} ({torch.cuda.get_device_name(0) if dev==\'cuda\' else \'CPU\'}):")\n    print(f"  Params: {count_params(raw_model):,}")\n    print(f"  Target Pollutants: {tr.n_poll} ({\'NO2\' if tr.n_poll==1 else \'NO2, CO, SO2\'})")\n    print(f"  Long-Mem Dropout: {cfg.get(\'long_mem_dropout\', 0.0):.2f}")\n    print(f"  Temporal Splits ({a.stage}): Train {cfg[\'splits\'][\'train\']}, Val {cfg[\'splits\'][\'val\']}, Test {cfg[\'splits\'][\'test\']}")\n    print(f"  Weather Augmented: {bool(a.use_weather)}")\n    print(f"  Structural Shift B: {bool(a.use_B)}")\n    print(f"  Slow S2 Encoder: {bool(a.use_s2)}")\n    print(f"  Multi-GPU DataParallel: {use_dp} ({num_gpus} devices)")\n\n    ema = copy.deepcopy(raw_model).eval()\n    nd = [p for n, p in raw_model.named_parameters() if p.ndim <= 1 or "prior.W" in n]\n    dd = [p for n, p in raw_model.named_parameters() if not (p.ndim <= 1 or "prior.W" in n)]\n    opt = torch.optim.AdamW([dict(params=dd, weight_decay=cfg["wd"]), dict(params=nd, weight_decay=0.0)], lr=cfg["lr"])\n    iters = cfg["samples_per_epoch"] // cfg["batch"]; total = iters * cfg["epochs"]; step = 0\n    sched = lambda s: min(1, (s + 1) / (cfg["warmup"] * iters)) * 0.5 * (1 + math.cos(math.pi * min(1, s / total)))\n    amp = dev == "cuda"\n    scaler = torch.cuda.amp.GradScaler(enabled=amp) if dev == "cuda" else torch.amp.GradScaler("cpu", enabled=False)\n    peak = torch.tensor(tr.peak_thr, device=dev)\n    os.makedirs(a.out, exist_ok=True); best, bad = 1e9, 0\n\n    for ep in range(cfg["epochs"]):\n        model.train(); raw_model.train(); run = 0.0\n        for b_idx, b in enumerate(tl):\n            for g in opt.param_groups: g["lr"] = cfg["lr"] * sched(step)\n            b = to_dev(b, dev)\n            with torch.cuda.amp.autocast(enabled=amp):\n                o = model(model_in(b))\n            o = {k: (v.float() if torch.is_tensor(v) else v) for k, v in o.items()}\n            loss = dcnet_loss(o, b["y"], b["ym"], b["ev"], peak_thr=peak)\n            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)\n            torch.nn.utils.clip_grad_norm_(raw_model.parameters(), cfg["clip"]); scaler.step(opt); scaler.update(); step += 1; run += loss.item()\n            with torch.no_grad():\n                d = min(cfg["ema"], (1 + step) / (10 + step))\n                for pe, pm in zip(ema.parameters(), raw_model.parameters()): pe.mul_(d).add_(pm.detach(), alpha=1 - d)\n            if (b_idx + 1) % 25 == 0 or (b_idx + 1) == iters:\n                print(f"  ep {ep:02d} [{b_idx+1:3d}/{iters}] batch loss: {loss.item():.4f} | lr: {opt.param_groups[0][\'lr\']:.2e}", flush=True)\n\n        vloss, r2, cov = evaluate(ema, vl, dev, peak, n_poll=tr.n_poll)\n        if tr.n_poll == 1:\n            print(f"ep {ep:02d} train {run/iters:.4f} val {vloss:.4f} | R2[h=5d,150d] NO2 {r2[0,0]:.3f}/{r2[-1,0]:.3f} | 80%-cov {cov.mean():.3f}", flush=True)\n        else:\n            print(f"ep {ep:02d} train {run/iters:.4f} val {vloss:.4f} | R2[h=5d,150d] NO2 {r2[0,0]:.3f}/{r2[-1,0]:.3f} CO {r2[0,1]:.3f}/{r2[-1,1]:.3f} | 80%-cov {cov.mean():.3f}", flush=True)\n\n        if vloss < best:\n            best, bad = vloss, 0\n            torch.save(dict(model=ema.state_dict(), norm=tr.norm.state(), s2=tr.s2_stats,\n                            wx=tr.wx_norm.state() if tr.wx_norm else None, cfg=cfg),\n                       os.path.join(a.out, "best.pt"))\n        else:\n            bad += 1\n            if bad >= cfg["patience"]:\n                print(f"[Early Stop] Triggered at epoch {ep} (patience={cfg[\'patience\']}). Preserving laptop hardware.")\n                break\n\n        if a.max_epochs and ep + 1 >= a.max_epochs:\n            print(f"Reached requested max_epochs={a.max_epochs}. Stopping.")\n            break\n\n        time.sleep(0.3)\n\n    print(f"Training completed successfully. Best val loss: {best:.4f}. Saved checkpoint to {os.path.join(a.out, \'best.pt\')}")\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--cache", required=True)\n    ap.add_argument("--out", default="runs/dcnet")\n    ap.add_argument("--cfg", default=None)\n    ap.add_argument("--seed", type=int, default=0)\n    ap.add_argument("--workers", type=int, default=2)\n    ap.add_argument("--batch", type=int, default=8)\n    ap.add_argument("--vram_cap", type=float, default=0.60)\n    ap.add_argument("--use_weather", type=int, default=1)\n    ap.add_argument("--use_B", type=int, default=1)\n    ap.add_argument("--use_s2", type=int, default=1)\n    ap.add_argument("--epochs", "--max_epochs", dest="max_epochs", type=int, default=0)\n    ap.add_argument("--stage", default="1", choices=["1", "2", "legacy"])\n    ap.add_argument("--poll", default="NO2", choices=["NO2", "all"])\n    ap.add_argument("--n_poll", type=int, default=1)\n    ap.add_argument("--long_mem_dropout", type=float, default=0.5)\n    ap.add_argument("--no_dp", action="store_true", help="Disable DataParallel even if multi-GPU is available")\n    main(ap.parse_args())\n')
print('Deployed: train.py')
with open('evaluate.py', 'w') as f:
    f.write('#!/usr/bin/env python3\n"""\nevaluate.py - Full multi-horizon evaluation for DC-Net\n======================================================\nComputes metrics per pollutant x horizon in physical units:\n  - R2 and Pearson r\n  - CSI (Critical Success Index) for plume peaks (> 90th pct)\n  - 80% coverage and interval width (q90 - q10)\n  - SO2 occurrence ROC-AUC and PR-AUC\n  - Skill over Harmonic Climatology and Persistence\n======================================================\n"""\nimport argparse, json, os, numpy as np, pandas as pd, torch\nfrom torch.utils.data import DataLoader\nfrom sklearn.metrics import roc_auc_score, average_precision_score\nfrom dataset import DCDataset, S5PNorm, WeatherNorm, collate\nfrom dcnet import DCNet, HORIZONS_DAYS, HORIZON_STEPS\n\ndef to_dev(b, dev): return {k: (v.to(dev, non_blocking=True) if torch.is_tensor(v) else v) for k, v in b.items()}\ndef model_in(b):\n    res = dict(fast_v=b["fast_v"], fast_m=b["fast_m"], s2_slow=b["s2_slow"], long_mem=b["long_mem"],\n               crop=b["crop"], doy_tgt=b["doy_tgt"], doy_org_sc=b["doy_org_sc"])\n    if "weather" in b: res["weather"] = b["weather"]\n    return res\n\ndef calc_csi(y_true, y_pred, thr):\n    hits = np.sum((y_pred >= thr) & (y_true >= thr))\n    false_alarms = np.sum((y_pred >= thr) & (y_true < thr))\n    misses = np.sum((y_pred < thr) & (y_true >= thr))\n    denom = hits + false_alarms + misses\n    return hits / max(denom, 1)\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--ckpt", required=True, help="Path to best.pt checkpoint")\n    p.add_argument("--cache", required=True, help="Path to cache dir")\n    p.add_argument("--split", default="test", choices=["val", "test"])\n    p.add_argument("--out", default="runs/dcnet_eval")\n    args = p.parse_args()\n\n    os.makedirs(args.out, exist_ok=True)\n    dev = "cuda" if torch.cuda.is_available() else "cpu"\n\n    ckpt = torch.load(args.ckpt, map_location=dev)\n    cfg = ckpt["cfg"]\n\n    # Reconstruct norms\n    s5_norm = S5PNorm()\n    s5_norm.s = ckpt["norm"]["s"]\n    s5_norm.mu = ckpt["norm"]["mu"]\n    s5_norm.sd = ckpt["norm"]["sd"]\n\n    wx_norm = None\n    if ckpt.get("wx") is not None:\n        wx_norm = WeatherNorm()\n        wx_norm.mu = ckpt["wx"]["mu"]\n        wx_norm.sd = ckpt["wx"]["sd"]\n\n    ds = DCDataset(args.cache, args.split, cfg, norm=s5_norm, s2_stats=ckpt["s2"], wx_norm=wx_norm)\n    dl = DataLoader(ds, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate)\n\n    use_wx = wx_norm is not None\n    model = DCNet(ds.n_poll, (ds.H5, ds.W5), use_B=bool(cfg.get("use_B", 1)), use_s2=bool(cfg.get("use_s2", 1)), use_weather=use_wx).to(dev)\n    model.load_state_dict(ckpt["model"])\n    model.eval()\n\n    all_poll_names = ["NO2", "CO", "SO2"]\n    poll_names = [all_poll_names[c] for c in ds.poll_idx]\n    Q = len(HORIZONS_DAYS)\n\n    # Compute Empirical Training-Set Pixel Climatology\n    tr_idx = [i for i in range(ds.T) if ds.ws[i] <= pd.Timestamp(cfg["splits"]["train"][1])]\n    clim_empirical = np.zeros((12, ds.n_poll, ds.H5, ds.W5), dtype=np.float32)\n    ws_months = np.array([d.month for d in ds.ws])\n    for m in range(1, 13):\n        m_idx = [i for i in tr_idx if ws_months[i] == m]\n        if len(m_idx) > 0:\n            for p_i, c in enumerate(ds.poll_idx):\n                v_sub = ds.val[m_idx, c]\n                m_sub = ds.msk[m_idx, c] > 0\n                cnt = m_sub.sum(axis=0)\n                with np.errstate(invalid="ignore", divide="ignore"):\n                    c_mean = np.where(cnt > 0, (v_sub * m_sub).sum(axis=0) / np.maximum(cnt, 1), 0.0)\n                clim_empirical[m - 1, p_i] = np.nan_to_num(c_mean, nan=0.0)\n\n    # Fill any missing pixel-months with overall training pixel mean\n    for p_i, c in enumerate(ds.poll_idx):\n        v_sub = ds.val[tr_idx, c]\n        m_sub = ds.msk[tr_idx, c] > 0\n        cnt_all = m_sub.sum(axis=0)\n        with np.errstate(invalid="ignore", divide="ignore"):\n            ov_mean = np.where(cnt_all > 0, (v_sub * m_sub).sum(axis=0) / np.maximum(cnt_all, 1), 0.0)\n        for m in range(12):\n            missing = clim_empirical[m, p_i] == 0.0\n            clim_empirical[m, p_i] = np.where(missing, ov_mean, clim_empirical[m, p_i])\n    \n    # Storage for accumulated predictions\n    preds_phys = {p_i: [[] for _ in range(Q)] for p_i in range(ds.n_poll)}\n    targets_phys = {p_i: [[] for _ in range(Q)] for p_i in range(ds.n_poll)}\n    masks = {p_i: [[] for _ in range(Q)] for p_i in range(ds.n_poll)}\n    q10_phys = {p_i: [[] for _ in range(Q)] for p_i in range(ds.n_poll)}\n    q90_phys = {p_i: [[] for _ in range(Q)] for p_i in range(ds.n_poll)}\n    clim_phys_list = {p_i: [[] for _ in range(Q)] for p_i in range(ds.n_poll)}\n    priors_phys = {p_i: [[] for _ in range(Q)] for p_i in range(ds.n_poll)}\n    persist_phys = {p_i: [[] for _ in range(Q)] for p_i in range(ds.n_poll)}\n    occ_logits = [[] for _ in range(Q)]\n    so2_events = [[] for _ in range(Q)]\n\n    print(f"Running evaluation on {len(ds)} {args.split} windows (Pollutants: {poll_names})...")\n    with torch.no_grad():\n        for b_idx, b in enumerate(dl):\n            b_dev = to_dev(b, dev)\n            if torch.is_tensor(b["crop"]):\n                if b["crop"].dim() == 2:\n                    H, W = int(b["crop"][0, 2]), int(b["crop"][0, 3])\n                    i0, j0 = int(b["crop"][0, 0]), int(b["crop"][0, 1])\n                else:\n                    H, W = int(b["crop"][2]), int(b["crop"][3])\n                    i0, j0 = int(b["crop"][0]), int(b["crop"][1])\n            else:\n                H, W = int(b["crop"][2]), int(b["crop"][3])\n                i0, j0 = int(b["crop"][0]), int(b["crop"][1])\n\n            o = model(model_in(b_dev))\n            crop = lambda t: t[..., :H, :W]\n            q50 = crop(o["q50"]).cpu().numpy()[0]\n            q10 = crop(o["q10"]).cpu().numpy()[0]\n            q90 = crop(o["q90"]).cpu().numpy()[0]\n            prior = crop(model.prior(b_dev["doy_tgt"], b["crop"], (H, W))).cpu().numpy()[0]\n            \n            y = crop(b["y"]).numpy()[0]\n            ym = crop(b["ym"]).numpy()[0]\n            \n            # Origin frame for persistence\n            # Last *valid* observation per pixel (not just the last window, which is 0/climatology where cloudy)\n            fv_all, fm_all = b["fast_v"][0].numpy(), b["fast_m"][0].numpy()          # (L, P, H, W)\n            t_idx = np.arange(fv_all.shape[0])[:, None, None, None]\n            last_i = np.where(fm_all > 0, t_idx, -1).max(0)                           # (P, H, W)\n            last_valid_fast = np.take_along_axis(fv_all, np.maximum(last_i, 0)[None], 0)[0][..., :H, :W]\n            has_hist = (last_i >= 0)[..., :H, :W]\n            \n            occ = crop(o["occ_logit"]).cpu().numpy()[0]\n            ev = crop(b["ev"]).numpy()[0]\n\n            orig_idx = ds.origins[b_idx]\n            steps = cfg["steps"]\n\n            for q in range(Q):\n                occ_logits[q].append(occ[q].flatten())\n                so2_events[q].append(ev[q].flatten())\n                st = steps[q]\n                tgt_ts = ds.ws[orig_idx] + pd.Timedelta(days=5 * st)\n                tgt_m = tgt_ts.month\n                clim_sub = clim_empirical[tgt_m - 1, :, i0:i0 + H, j0:j0 + W]\n\n                for p_i in range(ds.n_poll):\n                    c = ds.poll_idx[p_i]\n                    m = ym[q, p_i] > 0\n                    if not m.any(): continue\n                    masks[p_i][q].append(m.flatten())\n                    \n                    # Convert to physical units\n                    p_p = s5_norm.inv(q50[q, p_i], c)\n                    t_p = s5_norm.inv(y[q, p_i], c)\n                    l_p = s5_norm.inv(q10[q, p_i], c)\n                    u_p = s5_norm.inv(q90[q, p_i], c)\n                    pr_p = s5_norm.inv(prior[q, p_i], c)\n                    cl_p = clim_sub[p_i]\n                    pers_p = np.where(has_hist[p_i], s5_norm.inv(last_valid_fast[p_i], c), cl_p)\n                    \n                    preds_phys[p_i][q].append(p_p.flatten())\n                    targets_phys[p_i][q].append(t_p.flatten())\n                    q10_phys[p_i][q].append(l_p.flatten())\n                    q90_phys[p_i][q].append(u_p.flatten())\n                    priors_phys[p_i][q].append(pr_p.flatten())\n                    persist_phys[p_i][q].append(pers_p.flatten())\n                    clim_phys_list[p_i][q].append(cl_p.flatten())\n\n    results = []\n    for p_i, poll in enumerate(poll_names):\n        c = ds.poll_idx[p_i]\n        peak_thr_phys = float(s5_norm.inv(ds.peak_thr[p_i], c))\n        for q, h_days in enumerate(HORIZONS_DAYS):\n            if len(targets_phys[p_i][q]) == 0: continue\n            \n            m_all = np.concatenate(masks[p_i][q])\n            y_t = np.concatenate(targets_phys[p_i][q])[m_all]\n            y_p = np.concatenate(preds_phys[p_i][q])[m_all]\n            y_lo = np.concatenate(q10_phys[p_i][q])[m_all]\n            y_hi = np.concatenate(q90_phys[p_i][q])[m_all]\n            y_clim = np.concatenate(clim_phys_list[p_i][q])[m_all]\n            y_pr = np.concatenate(priors_phys[p_i][q])[m_all]\n            y_pers = np.concatenate(persist_phys[p_i][q])[m_all]\n\n            sse_m = np.sum((y_t - y_p) ** 2)\n            sse_clim = np.sum((y_t - y_clim) ** 2)\n            sse_pr = np.sum((y_t - y_pr) ** 2)\n            sse_pers = np.sum((y_t - y_pers) ** 2)\n            sst = np.sum((y_t - np.mean(y_t)) ** 2)\n\n            r2 = 1.0 - sse_m / max(sst, 1e-12)\n            r = np.corrcoef(y_t, y_p)[0, 1] if len(y_t) > 1 else np.nan\n            skill_clim = (1.0 - sse_m / max(sse_clim, 1e-12)) * 100.0\n            skill_pers = (1.0 - sse_m / max(sse_pers, 1e-12)) * 100.0\n            skill_prior = (1.0 - sse_m / max(sse_pr, 1e-12)) * 100.0\n            \n            cov = np.mean((y_t >= y_lo) & (y_t <= y_hi))\n            width = np.mean(y_hi - y_lo)\n            csi = calc_csi(y_t, y_p, peak_thr_phys)\n\n            row = dict(pollutant=poll, horizon_days=h_days, horizon_step=h_days//5,\n                       R2=float(r2), pearson_r=float(r), skill_vs_clim_pct=float(skill_clim),\n                       skill_vs_pers_pct=float(skill_pers), skill_vs_prior_pct=float(skill_prior),\n                       coverage_80=float(cov), interval_width=float(width), csi_p90=float(csi),\n                       n_obs=int(len(y_t)))\n            results.append(row)\n\n    df_res = pd.DataFrame(results)\n    csv_path = os.path.join(args.out, f"dcnet_{args.split}_metrics.csv")\n    df_res.to_csv(csv_path, index=False)\n    print(f"\\nSaved evaluation metrics to {csv_path}:")\n    if len(df_res) > 0:\n        cols_to_print = [c for c in ["pollutant", "horizon_days", "R2", "pearson_r", "skill_vs_clim_pct", "coverage_80", "csi_p90"] if c in df_res.columns]\n        print(df_res[cols_to_print].to_string())\n    else:\n        print("No evaluation records produced for this split.")\n\nif __name__ == "__main__":\n    main()\n')
print('Deployed: evaluate.py')
with open('make_cache.py', 'w') as f:
    f.write('"""Build the aligned training cache from your notebook\'s GeoTIFFs (run once, CPU only).\nOutputs (in --out): s5p_val.npy, s5p_mask.npy, s5p_month.npy, s5p_month_ok.npy, s2_month.npy (fp16, 8x grid),\ns2_month_ok.npy, weather_5d.npy, meta.json.  Regular 5-day window axis: missing windows are kept (mask=0), never dropped."""\nimport argparse, os, json, datetime as dt, numpy as np, pandas as pd, rasterio\nfrom concurrent.futures import ThreadPoolExecutor\nfrom tqdm import tqdm\nfrom align_tn import s5p_grid, align_s2_to_s5p, R\n\ndef windows(start, end, step=5):\n    cur, out = dt.datetime.strptime(start, "%Y-%m-%d"), []\n    end = dt.datetime.strptime(end, "%Y-%m-%d")\n    while cur + dt.timedelta(days=step) <= end:\n        out.append(cur); cur += dt.timedelta(days=step)\n    return out\n\ndef main(a):\n    os.makedirs(a.out, exist_ok=True)\n    W = windows(a.start, a.end); T = len(W)\n    ref = next(os.path.join(a.s5p_dir, f) for f in sorted(os.listdir(a.s5p_dir)) if f.startswith("s5p_2"))\n    tf, H5, W5, crs = s5p_grid(ref)\n    win = None\n\n    if getattr(a, "roi", None) == "chennai":\n        bbox = [79.95, 12.80, 80.45, 13.30]\n        with rasterio.open(ref) as s:\n            w = rasterio.windows.from_bounds(*bbox, transform=s.transform).round_offsets().round_shape()\n            c_off = max(0, min(int(w.col_off), s.width - 1))\n            r_off = max(0, min(int(w.row_off), s.height - 1))\n            w_w = min(int(w.width), s.width - c_off)\n            w_h = min(int(w.height), s.height - r_off)\n            win = rasterio.windows.Window(c_off, r_off, w_w, w_h)\n            tf = rasterio.windows.transform(win, s.transform)\n            H5, W5 = w_h, w_w\n            print(f"[ROI Active: Chennai] Cropped S5P grid to ({H5}, {W5}) from bbox {bbox}.")\n\n    val = np.zeros((T, 3, H5, W5), np.float32); msk = np.zeros((T, 3, H5, W5), np.uint8)\n    for i, w in enumerate(tqdm(W, desc="S5P")):\n        p = os.path.join(a.s5p_dir, f"s5p_{w:%Y%m%d}.tif")\n        if not os.path.exists(p): continue\n        with rasterio.open(p) as s:\n            x = s.read(window=win).astype(np.float32)[:, :H5, :W5] if win else s.read().astype(np.float32)[:, :H5, :W5]\n        ok = np.isfinite(x) & (x > 0 if a.zero_is_invalid else True)\n        val[i, :, :x.shape[1], :x.shape[2]] = np.where(ok, x, 0); msk[i, :, :x.shape[1], :x.shape[2]] = ok\n    months = pd.period_range(W[0], W[-1], freq="M"); M = len(months)\n    mon_of = np.array([months.get_loc(pd.Period(w, "M")) for w in W])\n    s5m = np.zeros((M, 3, H5, W5), np.float32); s5ok = np.zeros((M, 3, H5, W5), np.uint8)\n    for m in range(M):\n        ii = np.where(mon_of == m)[0]; c = msk[ii].sum(0)\n        s5m[m] = (val[ii] * msk[ii]).sum(0) / np.maximum(c, 1); s5ok[m] = c > 0\n\n    # Smart Resolution Selection & Cache Footprint Guard\n    is_state_wide = (H5 > 25 or W5 > 25)\n    if a.s2_ratio is not None and a.s2_ratio > 0:\n        R_use = a.s2_ratio\n    else:\n        # Full-state Tamil Nadu raw S2 export is ~600m native (1023x780) -> R=8 preserves 100% of spatial data 1:1 without artificial interpolation (~1.2 GB).\n        # Focused ROI (e.g. Chennai 13x13) has 100m native S2 -> R=32 yields ~300 MB.\n        R_use = 8 if is_state_wide else 32\n\n    H8, W8 = H5 * R_use, W5 * R_use\n    est_bytes = M * 12 * H8 * W8 * 2  # float16\n    est_gb = est_bytes / (1024 ** 3)\n    print(f"[Cache Pre-Flight] S5P Grid: ({H5}, {W5}) | S2 Ratio: {R_use}x | High-Res S2 Grid: ({H8}, {W8})")\n    print(f"[Cache Pre-Flight] Target s2_month.npy: {est_gb:.2f} GB ({est_bytes / (1024**2):.1f} MB)")\n\n    out_abs = os.path.abspath(a.out)\n    is_tmp_storage = ("/tmp" in out_abs)\n    safe_disk_limit = 55.0 if is_tmp_storage else 14.0\n    storage_type = "ephemeral /kaggle/tmp (~60 GB available)" if is_tmp_storage else "standard /kaggle/working (20 GB quota)"\n\n    if est_gb > safe_disk_limit and not getattr(a, "force_large_cache", False):\n        raise RuntimeError(\n            f"Calculated cache size ({est_gb:.2f} GB) exceeds safe disk limit ({safe_disk_limit} GB) for {storage_type}. "\n            f"Grid ({H5}x{W5}) with R={R_use} creates {H8}x{W8} ({H8*W8:,} pixels) per month.\\n"\n            f"Solutions:\\n"\n            f"  1. Use ephemeral storage: --out /kaggle/tmp/cache (up to ~60 GB available space);\\n"\n            f"  2. For state-wide grids, use --s2_ratio 12 (~2.9 GB, matching native 500m S2 export);\\n"\n            f"  3. For high-res urban study, pass --roi chennai with --s2_ratio 32 (only ~300 MB);\\n"\n            f"  4. To bypass this guard if disk allows, add --force_large_cache 1."\n        )\n\n    s2m = np.lib.format.open_memmap(os.path.join(a.out, "s2_month.npy"), "w+", np.float16, (M, 12, H8, W8))\n    s2ok = np.lib.format.open_memmap(os.path.join(a.out, "s2_month_ok.npy"), "w+", np.uint8, (M, H8, W8))\n\n    def align_worker(item):\n        i, w_date = item\n        p = os.path.join(a.s2_dir, f"s2_{w_date:%Y%m%d}.tif")\n        if not os.path.exists(p): return None\n        x, v = align_s2_to_s5p(p, tf, H5, W5, crs, r=R_use); ok = v[0] > 0.5\n        x = np.clip(x, -100.0, 10000.0)\n        x[:, ~ok] = np.nan\n        with np.errstate(all="ignore"):\n            return x.astype(np.float16), ok\n\n    workers = getattr(a, "workers", 4) or 4\n    with ThreadPoolExecutor(max_workers=workers) as executor:\n        for m in tqdm(range(M), desc=f"S2 monthly composites ({workers} workers)"):\n            items = [(i, W[i]) for i in np.where(mon_of == m)[0]]\n            results = list(executor.map(align_worker, items))\n            stack = [r[0] for r in results if r is not None]\n            if stack:\n                with np.errstate(all="ignore"):\n                    med = np.nanmedian(np.stack(stack).astype(np.float32), axis=0)\n                    ok = np.isfinite(med[0])\n                    s2m[m] = np.nan_to_num(med, nan=0.0).astype(np.float16)\n                    s2ok[m] = ok\n\n    for k, v in dict(s5p_val=val, s5p_mask=msk, s5p_month=s5m, s5p_month_ok=s5ok).items(): np.save(os.path.join(a.out, k + ".npy"), v)\n    s2m.flush(); s2ok.flush()\n\n    # Weather 5-day window extraction\n    # Prefer era5_tn_clusters.csv for multi-cluster state-wide regional representation over Chennai-only era5_daily.csv\n    wx_clusters_path = os.path.join(os.path.dirname(a.s5p_dir), "..", "weather", "era5_tn_clusters.csv")\n    wx_default_path = os.path.join(os.path.dirname(a.s5p_dir), "..", "weather", "era5_daily.csv")\n    wx_path = a.weather if (hasattr(a, "weather") and a.weather) else (wx_clusters_path if os.path.exists(wx_clusters_path) else wx_default_path)\n\n    if os.path.exists(wx_path):\n        df_wx = pd.read_csv(wx_path)\n        df_wx["date"] = pd.to_datetime(df_wx["date"])\n        wx_cols = ["blh", "u10", "v10", "wind_speed", "tp", "t2m", "rh"]\n\n        # If multi-cluster dataset, aggregate across all regional clusters per day\n        if "cluster" in df_wx.columns:\n            print(f"Aggregating {df_wx[\'cluster\'].nunique()} regional clusters from {wx_path}...")\n            df_wx = df_wx.groupby("date")[wx_cols].mean().reset_index()\n\n        # Handle missing 2024 blh values via causal DOY climatology imputation (strictly from training data)\n        if df_wx["blh"].isna().any():\n            nan_cnt = df_wx["blh"].isna().sum()\n            print(f"Causally imputing {nan_cnt} missing ERA5 \'blh\' values using DOY training climatology...")\n            tr_dates = df_wx[df_wx["date"] <= "2023-12-31"]\n            doy_clim = tr_dates.groupby(tr_dates["date"].dt.dayofyear)["blh"].mean()\n\n            for idx in df_wx[df_wx["blh"].isna()].index:\n                d = df_wx.loc[idx, "date"].dayofyear\n                if d in doy_clim and np.isfinite(doy_clim[d]):\n                    val = doy_clim[d]\n                else:\n                    nearby = doy_clim[(doy_clim.index >= d - 15) & (doy_clim.index <= d + 15)]\n                    val = nearby.mean() if len(nearby) > 0 else tr_dates["blh"].mean()\n                df_wx.loc[idx, "blh"] = val\n\n        wx_5d = np.zeros((T, len(wx_cols) + 1), np.float32)\n        for i, w in enumerate(W):\n            w_end = w + dt.timedelta(days=5)\n            sub = df_wx[(df_wx["date"] >= w) & (df_wx["date"] < w_end)]\n            if len(sub) > 0:\n                vals = sub[wx_cols].mean().values\n                vent = vals[0] * vals[3]\n                wx_5d[i] = np.append(vals, vent)\n\n        # Final check: ensure 0 NaNs remain\n        if np.isnan(wx_5d).any():\n            col_means = np.nanmean(wx_5d, axis=0)\n            nan_rows = np.where(np.isnan(wx_5d).any(axis=1))[0]\n            print(f"Warning: Imputing {len(nan_rows)} windows with remaining NaNs.")\n            for r in nan_rows:\n                for c_i in range(wx_5d.shape[1]):\n                    if np.isnan(wx_5d[r, c_i]):\n                        wx_5d[r, c_i] = col_means[c_i]\n\n        np.save(os.path.join(a.out, "weather_5d.npy"), wx_5d)\n        print("weather cache ready:", wx_5d.shape, "NaNs remaining:", int(np.isnan(wx_5d).sum()))\n\n    json.dump(dict(T=T, M=M, H5=H5, W5=W5, H8=H8, W8=W8, s2_ratio=R_use, win_start=[f"{w:%Y-%m-%d}" for w in W],\n                   month_start=[f"{p.start_time:%Y-%m-%d}" for p in months], month_end=[f"{p.end_time:%Y-%m-%d}" for p in months]),\n              open(os.path.join(a.out, "meta.json"), "w"))\n    print("cache ready:", T, "windows,", M, "months, S5P grid", (H5, W5), "S2 grid", (H8, W8), f"(ratio {R_use}x)")\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--s2_dir"); ap.add_argument("--s5p_dir"); ap.add_argument("--out")\n    ap.add_argument("--weather", default=None)\n    ap.add_argument("--workers", type=int, default=4, help="Number of CPU worker threads for raster alignment (default 4)")\n    ap.add_argument("--s2_ratio", type=int, default=0, help="S2 subgrid ratio per S5P cell (0=auto: 32 for Chennai ROI ~300MB, 8 for full TN ~1.2GB native 600m 1:1)")\n    ap.add_argument("--roi", default="none", choices=["none", "chennai"], help="Crop ROI region (e.g. \'chennai\' for high-res urban study)")\n    ap.add_argument("--force_large_cache", type=int, default=0, help="Bypass the 14GB safety check if disk space allows")\n    ap.add_argument("--start", default="2019-01-01"); ap.add_argument("--end", default="2024-12-31")\n    ap.add_argument("--zero_is_invalid", type=int, default=1)\n    main(ap.parse_args())\n')
print('Deployed: make_cache.py')
with open('app.py', 'w') as f:
    f.write('#!/usr/bin/env python3\n"""\nAirQ-CV: Dual-Clock DC-Net Interactive Inference Dashboard\n==========================================================\nInteractive Gradio Service for Spatial-Temporal Pollution Forecasts\nFeatures:\n  - Labeled dataset split navigation: Train (2019-2023), Val (2024), Test (2025-2026)\n  - Interactive Date / Step sliders across 5-day observation windows\n  - Multi-Horizon forecast selection (h = 5d to 150d)\n  - Side-by-side 2D Spatial Comparison (True S5P vs Model Pred vs High-Res Fine Disaggregation)\n  - Physical unit inversion (mol/m^2) matching Sentinel-5P standard observations\n  - Uncertainty interval visualization (Q10 - Q50 - Q90) and error heatmaps\n  - Built-in Demo / Synthetic Mode if checkpoint or cache is not yet available\n==========================================================\n"""\n\nimport argparse, io, os, math, json, sys\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport matplotlib.gridspec as gridspec\nfrom PIL import Image\n\nimport torch\nimport torch.nn.functional as F\n\ntry:\n    import gradio as gr\n    HAS_GRADIO = True\nexcept ImportError:\n    HAS_GRADIO = False\n\n# Import model components if available in path\nsys.path.append(os.path.dirname(os.path.abspath(__file__)))\ntry:\n    from dataset import DCDataset, S5PNorm, WeatherNorm, collate\n    from dcnet import DCNet, HORIZONS_DAYS, HORIZON_STEPS\n    HAS_DCNET = True\nexcept ImportError:\n    HAS_DCNET = False\n    HORIZONS_DAYS = (5, 10, 15, 20, 25, 30, 60, 150)\n    HORIZON_STEPS = (1, 2, 3, 4, 5, 6, 12, 30)\n\nHORIZON_LABELS = [f"+{d} days (Step #{s})" for d, s in zip(HORIZONS_DAYS, HORIZON_STEPS)]\nSPLITS = {\n    "Train (2019–2023)": "train",\n    "Validation (2024 Held-Out)": "val",\n    "Test (2025–2026)": "test"\n}\n\n\n# ==============================================================================\n# 1. INFERENCE ENGINE (LIVE CHECKPOINT + SYNTHETIC FALLBACK)\n# ==============================================================================\nclass InferenceEngine:\n    def __init__(self, ckpt_path=None, cache_dir=None, dev="cpu"):\n        self.ckpt_path = ckpt_path\n        self.cache_dir = cache_dir\n        self.dev = dev\n        self.model = None\n        self.datasets = {}\n        self.is_live = False\n        self.n_poll = 1\n        self.poll_names = ["NO2"]\n        self.s5_norm = None\n\n        if ckpt_path and os.path.exists(ckpt_path) and cache_dir and os.path.exists(cache_dir) and HAS_DCNET:\n            try:\n                self._load_live_model()\n                self.is_live = True\n                print(f"[Engine] Successfully loaded live checkpoint from {ckpt_path} on {dev}.")\n            except Exception as e:\n                print(f"[Engine Warning] Failed to load live model ({e}). Falling back to simulation mode.")\n                self.is_live = False\n        else:\n            print("[Engine] No active checkpoint/cache found. Running in high-fidelity simulation mode.")\n\n    def _load_live_model(self):\n        ckpt = torch.load(self.ckpt_path, map_location=self.dev)\n        cfg = ckpt.get("cfg", {})\n\n        # Restore normalizer\n        self.s5_norm = S5PNorm()\n        self.s5_norm.s = ckpt["norm"]["s"]\n        self.s5_norm.mu = ckpt["norm"]["mu"]\n        self.s5_norm.sd = ckpt["norm"]["sd"]\n\n        wx_norm = None\n        if ckpt.get("wx") is not None:\n            wx_norm = WeatherNorm()\n            wx_norm.mu = ckpt["wx"]["mu"]\n            wx_norm.sd = ckpt["wx"]["sd"]\n\n        # Instantiate datasets for each split\n        for split_label, split_key in SPLITS.items():\n            try:\n                self.datasets[split_key] = DCDataset(\n                    self.cache_dir, split_key, cfg,\n                    norm=self.s5_norm, s2_stats=ckpt.get("s2"), wx_norm=wx_norm\n                )\n            except Exception as e:\n                print(f"[Engine] Split {split_key} load warning: {e}")\n\n        sample_ds = next(iter(self.datasets.values()))\n        self.n_poll = sample_ds.n_poll\n        all_poll_names = ["NO2", "CO", "SO2"]\n        self.poll_names = [all_poll_names[c] for c in sample_ds.poll_idx]\n        self.H5, self.W5 = sample_ds.H5, sample_ds.W5\n        self.s2_ratio = sample_ds.s2_ratio\n\n        # Load DC-Net\n        use_wx = wx_norm is not None\n        self.model = DCNet(\n            self.n_poll, (self.H5, self.W5),\n            use_B=bool(cfg.get("use_B", 1)),\n            use_s2=bool(cfg.get("use_s2", 1)),\n            use_weather=use_wx\n        ).to(self.dev)\n        self.model.load_state_dict(ckpt["model"])\n        self.model.eval()\n\n    def get_split_info(self, split_key):\n        if self.is_live and split_key in self.datasets:\n            ds = self.datasets[split_key]\n            n_samples = len(ds.origins)\n            date_strings = [f"{ds.ws[o]:%Y-%m-%d} to {ds.we[o]:%Y-%m-%d}" for o in ds.origins]\n            return n_samples, date_strings\n        else:\n            # Synthetic split definitions\n            counts = {"train": 365, "val": 73, "test": 73}\n            n_samples = counts.get(split_key, 73)\n            base_years = {"train": 2021, "val": 2024, "test": 2025}\n            y = base_years.get(split_key, 2024)\n            date_strings = [f"{pd.Timestamp(f\'{y}-01-01\') + pd.Timedelta(days=5*i):%Y-%m-%d}" for i in range(n_samples)]\n            return n_samples, date_strings\n\n    @torch.no_grad()\n    def predict(self, split_key, sample_idx, horizon_step_idx, poll_idx=0):\n        """\n        Returns physical units (mol/m^2) for true, coarse_pred, fine_pred, q10, q90, biophysical\n        """\n        if self.is_live and split_key in self.datasets:\n            ds = self.datasets[split_key]\n            sample_idx = max(0, min(sample_idx, len(ds.origins) - 1))\n            b_raw = ds[sample_idx]\n            b = collate([b_raw])\n            b = {k: (v.to(self.dev) if torch.is_tensor(v) else v) for k, v in b.items()}\n\n            # Run DC-Net model\n            H, W = int(b["crop"][0, 2]), int(b["crop"][0, 3])\n            m_in = dict(fast_v=b["fast_v"], fast_m=b["fast_m"], s2_slow=b["s2_slow"],\n                        long_mem=b["long_mem"], crop=b["crop"], doy_tgt=b["doy_tgt"],\n                        doy_org_sc=b["doy_org_sc"])\n            if "weather" in b: m_in["weather"] = b["weather"]\n\n            out = self.model(m_in)\n\n            # Invert normalized S5P scale to physical concentrations\n            c = ds.poll_idx[poll_idx]\n            q50_z = out["q50"][0, horizon_step_idx, poll_idx, :H, :W].cpu().numpy()\n            q10_z = out["q10"][0, horizon_step_idx, poll_idx, :H, :W].cpu().numpy()\n            q90_z = out["q90"][0, horizon_step_idx, poll_idx, :H, :W].cpu().numpy()\n            y_z = b["y"][0, horizon_step_idx, poll_idx, :H, :W].cpu().numpy()\n            ym = b["ym"][0, horizon_step_idx, poll_idx, :H, :W].cpu().numpy()\n\n            q50 = self.s5_norm.inv(q50_z, c)\n            q10 = self.s5_norm.inv(q10_z, c)\n            q90 = self.s5_norm.inv(q90_z, c)\n            y_true = np.where(ym > 0, self.s5_norm.inv(y_z, c), np.nan)\n\n            # High-res fine disaggregation\n            r = ds.s2_ratio\n            if "fine_q50" in out:\n                fine_z = out["fine_q50"][0, horizon_step_idx, poll_idx, :H*r, :W*r].cpu().numpy()\n                fine_pred = self.s5_norm.inv(fine_z, c)\n            else:\n                fine_pred = np.repeat(np.repeat(q50, r, axis=0), r, axis=1)\n\n            # Surface context (NDBI index from S2 channel 12)\n            s2_data = b["s2_slow"][0].cpu().numpy()\n            biophysical = s2_data[12, :H*r, :W*r] if s2_data.shape[0] > 12 else np.zeros((H*r, W*r))\n\n            origin_date = f"{ds.ws[ds.origins[sample_idx]]:%Y-%m-%d}"\n            target_date = f"{ds.ws[ds.origins[sample_idx]] + pd.Timedelta(days=5*HORIZON_STEPS[horizon_step_idx]):%Y-%m-%d}"\n\n            return dict(\n                y_true=y_true, q50=q50, fine_q50=fine_pred, q10=q10, q90=q90,\n                biophysical=biophysical, origin_date=origin_date, target_date=target_date,\n                pollutant=self.poll_names[poll_idx]\n            )\n\n        else:\n            # High-fidelity synthetic plume simulation matching physical NO2 distributions (1.0e-5 to 7.0e-5 mol/m^2)\n            np.random.seed(sample_idx * 100 + horizon_step_idx * 10 + poll_idx)\n            H, W = 13, 13\n            r = 32\n            H_fine, W_fine = H * r, W * r\n\n            # Background atmospheric baseline\n            base = 2.0e-5 + 0.5e-5 * math.sin(sample_idx * 0.2)\n\n            # Synthesize 2 industrial plumes (e.g. Ennore Port and Manali industrial corridor)\n            yy, xx = np.mgrid[:H, :W]\n            p1 = 4.5e-5 * np.exp(-((xx - 9)**2 + (yy - 2)**2) / 2.5)\n            p2 = 3.0e-5 * np.exp(-((xx - 4)**2 + (yy - 8)**2) / 3.0)\n            noise_s5p = np.random.normal(0, 0.3e-5, (H, W))\n\n            y_true = np.clip(base + p1 + p2 + noise_s5p, 0.8e-5, 7.5e-5)\n\n            # Model prediction (smooth regional forecast)\n            pred_noise = np.random.normal(0, 0.25e-5, (H, W))\n            q50 = np.clip(base + p1 * 0.95 + p2 * 0.88 + pred_noise, 0.8e-5, 7.5e-5)\n            q10 = np.clip(q50 - 0.7e-5, 0.5e-5, None)\n            q90 = q50 + 0.8e-5\n\n            # High-resolution fine disaggregation with road networks & stack plume dispersion\n            yy_f, xx_f = np.mgrid[:H_fine, :W_fine]\n            p1_f = 5.2e-5 * np.exp(-((xx_f - 9*r)**2 + (yy_f - 2*r)**2) / (r * 1.8)**2)\n            p2_f = 3.8e-5 * np.exp(-((xx_f - 4*r)**2 + (yy_f - 8*r)**2) / (r * 2.2)**2)\n            # High-frequency road corridor emission modulation\n            arterial_road = 0.8e-5 * np.exp(-((yy_f - xx_f * 0.8)**2) / (r * 0.4)**2)\n            fine_pred = np.clip(base + p1_f + p2_f + arterial_road + np.random.normal(0, 0.15e-5, (H_fine, W_fine)), 0.8e-5, 8.5e-5)\n\n            # Simulated NDBI (built-up impervious index in [-0.5, 0.6])\n            biophysical = np.clip(np.sin(xx_f / 50.0) * np.cos(yy_f / 50.0) + (fine_pred - base) / 3e-5, -0.6, 0.8)\n\n            dates = [pd.Timestamp("2024-01-01") + pd.Timedelta(days=5 * sample_idx)]\n            origin_date = f"{dates[0]:%Y-%m-%d}"\n            target_date = f"{dates[0] + pd.Timedelta(days=5 * HORIZON_STEPS[horizon_step_idx]):%Y-%m-%d}"\n\n            return dict(\n                y_true=y_true, q50=q50, fine_q50=fine_pred, q10=q10, q90=q90,\n                biophysical=biophysical, origin_date=origin_date, target_date=target_date,\n                pollutant="NO2"\n            )\n\n\n# ==============================================================================\n# 2. HIGH-AESTHETIC DASHBOARD RENDERING FUNCTION\n# ==============================================================================\ndef render_dashboard(engine, split_label, sample_idx, horizon_label, show_fine, show_diff, show_biophysical):\n    split_key = SPLITS.get(split_label, "val")\n    horizon_idx = HORIZON_LABELS.index(horizon_label) if horizon_label in HORIZON_LABELS else 0\n\n    res = engine.predict(split_key, int(sample_idx), horizon_idx)\n\n    y_true = res["y_true"]\n    q50 = res["q50"]\n    fine_q50 = res["fine_q50"]\n    q10 = res["q10"]\n    q90 = res["q90"]\n    biophysical = res["biophysical"]\n    poll = res["pollutant"]\n\n    # Calculate metrics\n    valid_mask = np.isfinite(y_true)\n    if valid_mask.sum() > 2:\n        yt_v = y_true[valid_mask]\n        qp_v = q50[valid_mask]\n        mae = float(np.mean(np.abs(qp_v - yt_v)))\n        rmse = float(np.sqrt(np.mean((qp_v - yt_v)**2)))\n        r_corr = float(np.corrcoef(yt_v, qp_v)[0, 1]) if np.std(yt_v) > 0 and np.std(qp_v) > 0 else 0.0\n        sst = np.sum((yt_v - np.mean(yt_v))**2)\n        sse = np.sum((yt_v - qp_v)**2)\n        r2 = float(1.0 - sse / max(sst, 1e-12))\n        inside_cov = float(np.mean((yt_v >= q10[valid_mask]) & (yt_v <= q90[valid_mask])) * 100.0)\n    else:\n        mae, rmse, r_corr, r2, inside_cov = 0.0, 0.0, 0.0, 0.0, 80.0\n\n    # Color scaling: shared min/max across True and Pred for absolute fidelity\n    finite_vals = np.concatenate([y_true[np.isfinite(y_true)], q50])\n    vmin = float(np.percentile(finite_vals, 2))\n    vmax = float(np.percentile(finite_vals, 98))\n\n    # Determine plot layout\n    n_cols = 2\n    if show_fine: n_cols += 1\n    if show_diff: n_cols += 1\n    if show_biophysical: n_cols += 1\n\n    fig = plt.figure(figsize=(4.5 * n_cols, 4.8), dpi=140, facecolor="#0f172a")\n    gs = gridspec.GridSpec(1, n_cols, wspace=0.35, top=0.84, bottom=0.12, left=0.06, right=0.94)\n\n    curr_col = 0\n    # Panel 1: True S5P\n    ax1 = fig.add_subplot(gs[0, curr_col])\n    im1 = ax1.imshow(y_true, cmap="viridis", vmin=vmin, vmax=vmax, interpolation="nearest")\n    ax1.set_title(f"True {poll} (Sentinel-5P)", color="white", fontsize=11, fontweight="bold", pad=8)\n    ax1.set_ylabel(f"{split_label}\\nStep #{int(sample_idx)}", color="#94a3b8", fontsize=10, fontweight="bold")\n    ax1.tick_params(colors="#64748b", labelsize=8)\n    cb1 = fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)\n    cb1.ax.tick_params(colors="#94a3b8", labelsize=7)\n    cb1.formatter.set_powerlimits((0, 0))\n    cb1.ax.yaxis.get_offset_text().set_color("#94a3b8")\n    cb1.ax.yaxis.get_offset_text().set_fontsize(7)\n    curr_col += 1\n\n    # Panel 2: Pred Regional (Coarse)\n    ax2 = fig.add_subplot(gs[0, curr_col])\n    im2 = ax2.imshow(q50, cmap="viridis", vmin=vmin, vmax=vmax, interpolation="bilinear")\n    ax2.set_title(f"Pred {poll} (Model Coarse)", color="white", fontsize=11, fontweight="bold", pad=8)\n    ax2.tick_params(colors="#64748b", labelsize=8)\n    cb2 = fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)\n    cb2.ax.tick_params(colors="#94a3b8", labelsize=7)\n    cb2.formatter.set_powerlimits((0, 0))\n    cb2.ax.yaxis.get_offset_text().set_color("#94a3b8")\n    cb2.ax.yaxis.get_offset_text().set_fontsize(7)\n    curr_col += 1\n\n    # Panel 3 (Optional): High-Res Fine Disaggregation\n    if show_fine:\n        ax3 = fig.add_subplot(gs[0, curr_col])\n        im3 = ax3.imshow(fine_q50, cmap="viridis", vmin=vmin, vmax=vmax * 1.05, interpolation="bilinear")\n        ax3.set_title(f"Rank 1 Fine Disaggregation (~100m)", color="#38bdf8", fontsize=11, fontweight="bold", pad=8)\n        ax3.tick_params(colors="#64748b", labelsize=8)\n        cb3 = fig.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)\n        cb3.ax.tick_params(colors="#94a3b8", labelsize=7)\n        cb3.formatter.set_powerlimits((0, 0))\n        cb3.ax.yaxis.get_offset_text().set_color("#94a3b8")\n        cb3.ax.yaxis.get_offset_text().set_fontsize(7)\n        curr_col += 1\n\n    # Panel 4 (Optional): Difference Error Map\n    if show_diff:\n        ax4 = fig.add_subplot(gs[0, curr_col])\n        diff = q50 - y_true\n        diff_max = float(np.nanpercentile(np.abs(diff), 98)) + 1e-9\n        im4 = ax4.imshow(diff, cmap="coolwarm", vmin=-diff_max, vmax=diff_max, interpolation="nearest")\n        ax4.set_title(f"Residual Error (Pred - True)", color="#f87171", fontsize=11, fontweight="bold", pad=8)\n        ax4.tick_params(colors="#64748b", labelsize=8)\n        cb4 = fig.colorbar(im4, ax=ax4, fraction=0.046, pad=0.04)\n        cb4.ax.tick_params(colors="#94a3b8", labelsize=7)\n        cb4.formatter.set_powerlimits((0, 0))\n        cb4.ax.yaxis.get_offset_text().set_color("#94a3b8")\n        cb4.ax.yaxis.get_offset_text().set_fontsize(7)\n        curr_col += 1\n\n    # Panel 5 (Optional): Biophysical Index Context\n    if show_biophysical:\n        ax5 = fig.add_subplot(gs[0, curr_col])\n        im5 = ax5.imshow(biophysical, cmap="magma", interpolation="bilinear")\n        ax5.set_title(f"Sentinel-2 NDBI (Built-up)", color="#f59e0b", fontsize=11, fontweight="bold", pad=8)\n        ax5.tick_params(colors="#64748b", labelsize=8)\n        cb5 = fig.colorbar(im5, ax=ax5, fraction=0.046, pad=0.04)\n        cb5.ax.tick_params(colors="#94a3b8", labelsize=7)\n        curr_col += 1\n\n    # Header info\n    fig.suptitle(\n        f"AirQ-CV Forecast Dashboard | Origin: {res[\'origin_date\']}  -->  Target: {res[\'target_date\']} ({horizon_label})",\n        color="white", fontsize=13, fontweight="bold", y=0.97\n    )\n\n    buf = io.BytesIO()\n    fig.savefig(buf, format="png", bbox_inches="tight", facecolor=fig.get_facecolor(), edgecolor="none")\n    plt.close(fig)\n    buf.seek(0)\n    image_out = Image.open(buf)\n\n    # Metrics markdown table\n    metrics_md = f"""\n### 📊 Performance Scorecard (Snapshot at {res[\'target_date\']})\n| Metric | Value | Interpretation |\n| :--- | :--- | :--- |\n| **Spatial $R^2$ Score** | **`{r2:.3f}`** | Regional spatial correlation across pollution corridor |\n| **Pearson Correlation ($r$)** | **`{r_corr:.3f}`** | Pattern fidelity of industrial plume contours |\n| **Mean Absolute Error (MAE)** | **`{mae:.2e} mol/m²`** | Pixel-level absolute error in physical SI units |\n| **Root Mean Squared Error (RMSE)**| **`{rmse:.2e} mol/m²`** | Peak pollution penalty weighting |\n| **80% Prediction Interval Coverage**| **`{inside_cov:.1f}%`** | Calibration reliability ($Q_{{10}} \\\\le Y \\\\le Q_{{90}}$) |\n"""\n    return image_out, metrics_md\n\n\n# ==============================================================================\n# 3. GRADIO APP BUILDER\n# ==============================================================================\ndef build_app(ckpt_path=None, cache_dir=None):\n    engine = InferenceEngine(ckpt_path, cache_dir)\n\n    theme = gr.themes.Soft(\n        primary_hue="blue",\n        secondary_hue="slate",\n        neutral_hue="slate",\n    ).set(\n        body_background_fill="#0b0f19",\n        block_background_fill="#111827",\n        block_border_width="1px",\n        block_border_color="#1f2937",\n        button_primary_background_fill="#2563eb",\n        button_primary_text_color="#ffffff",\n    )\n\n    with gr.Blocks(title="AirQ-CV Inference Dashboard", theme=theme) as demo:\n        gr.Markdown(\n            """\n            # 🛰️ AirQ-CV: Dual-Clock DC-Net Satellite Pollution Dashboard\n            ### High-Resolution Multi-Horizon Forecasts with Biophysical Sentinel-2 Disaggregation\n            *Seamlessly navigate and evaluate forecasts across **Training**, **Validation (Held-Out)**, and **Testing** splits.*\n            """\n        )\n\n        with gr.Row():\n            with gr.Column(scale=1, min_width=320):\n                gr.Markdown("### ⚙️ Navigation & Forecast Controls")\n\n                split_radio = gr.Radio(\n                    choices=list(SPLITS.keys()),\n                    value="Validation (2024 Held-Out)",\n                    label="📂 Dataset Split Partition",\n                    info="Select temporal partition (Train: 2019-2023, Val: 2024 held-out, Test: 2025-2026)"\n                )\n\n                n_init, dates_init = engine.get_split_info("val")\n\n                step_slider = gr.Slider(\n                    minimum=0,\n                    maximum=max(1, n_init - 1),\n                    value=min(11, n_init - 1),\n                    step=1,\n                    label="⏱️ Observation Date / 5-Day Step Slider",\n                    info=f"Step index (Total: {n_init} five-day windows)"\n                )\n\n                with gr.Row():\n                    prev_btn = gr.Button("◀ Prev Step", size="sm")\n                    next_btn = gr.Button("Next Step ▶", size="sm")\n\n                horizon_dropdown = gr.Dropdown(\n                    choices=HORIZON_LABELS,\n                    value=HORIZON_LABELS[0],\n                    label="🎯 Forecast Lead Time Horizon",\n                    info="Prediction horizon from 5 days up to 150 days ahead"\n                )\n\n                gr.Markdown("### 🎨 Visualization Toggles")\n                show_fine_cb = gr.Checkbox(value=True, label="Show Rank 1 Fine Disaggregation (~100m)")\n                show_diff_cb = gr.Checkbox(value=True, label="Show Residual Error Heatmap (|Pred - True|)")\n                show_bio_cb = gr.Checkbox(value=True, label="Show Sentinel-2 Biophysical NDBI Context")\n\n                run_btn = gr.Button("🚀 Refresh Snapshot View", variant="primary", size="lg")\n\n            with gr.Column(scale=3):\n                plot_output = gr.Image(label="2D Spatial Pollution Contour Comparison", type="pil")\n                metrics_output = gr.Markdown()\n\n        # Update slider limits when split is changed\n        def on_split_change(split_label):\n            s_key = SPLITS.get(split_label, "val")\n            n_samp, _ = engine.get_split_info(s_key)\n            new_max = max(1, n_samp - 1)\n            default_val = min(11, new_max)\n            return gr.update(maximum=new_max, value=default_val, info=f"Step index (Total: {n_samp} windows)")\n\n        split_radio.change(on_split_change, inputs=[split_radio], outputs=[step_slider])\n\n        # Step navigation buttons\n        def on_prev(cur_val): return max(0, int(cur_val) - 1)\n        def on_next(cur_val, split_label):\n            s_key = SPLITS.get(split_label, "val")\n            n_samp, _ = engine.get_split_info(s_key)\n            return min(n_samp - 1, int(cur_val) + 1)\n\n        prev_btn.click(on_prev, inputs=[step_slider], outputs=[step_slider])\n        next_btn.click(on_next, inputs=[step_slider, split_radio], outputs=[step_slider])\n\n        # Core render trigger\n        render_inputs = [\n            split_radio, step_slider, horizon_dropdown,\n            show_fine_cb, show_diff_cb, show_bio_cb\n        ]\n\n        def trigger_render(s_rad, s_step, h_drop, fine, diff, bio):\n            return render_dashboard(engine, s_rad, s_step, h_drop, fine, diff, bio)\n\n        run_btn.click(trigger_render, inputs=render_inputs, outputs=[plot_output, metrics_output])\n        step_slider.change(trigger_render, inputs=render_inputs, outputs=[plot_output, metrics_output])\n        horizon_dropdown.change(trigger_render, inputs=render_inputs, outputs=[plot_output, metrics_output])\n        show_fine_cb.change(trigger_render, inputs=render_inputs, outputs=[plot_output, metrics_output])\n        show_diff_cb.change(trigger_render, inputs=render_inputs, outputs=[plot_output, metrics_output])\n        show_bio_cb.change(trigger_render, inputs=render_inputs, outputs=[plot_output, metrics_output])\n\n        # Initial launch trigger on load\n        demo.load(trigger_render, inputs=render_inputs, outputs=[plot_output, metrics_output])\n\n    return demo\n\n\n# ==============================================================================\n# 4. ENTRYPOINT\n# ==============================================================================\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description="AirQ-CV Gradio Inference UI")\n    parser.add_argument("--ckpt", default="runs/dcnet_main/best.pt", help="Path to best.pt checkpoint")\n    parser.add_argument("--cache", default="cache", help="Path to cache directory")\n    parser.add_argument("--port", type=int, default=7860, help="Local server port")\n    parser.add_argument("--share", action="store_true", help="Generate public Gradio link (ideal for Kaggle/Colab)")\n    args = parser.parse_args()\n\n    if not HAS_GRADIO:\n        print("Error: Gradio is not installed. Install with `pip install gradio`.")\n        sys.exit(1)\n\n    app = build_app(args.ckpt, args.cache)\n    print(f"\\n[AirQ-CV UI] Launching Gradio Dashboard on port {args.port} (share={args.share})...")\n    app.launch(server_name="0.0.0.0", server_port=args.port, share=args.share)\n')
print('Deployed: app.py')
with open('check_cache.py', 'w') as f:
    f.write('"""Valid-pixel / coverage audit of a DC-Net cache (run after make_cache.py).\n\nUsage:  python check_cache.py --cache /kaggle/tmp/cache [--min_valid 0.10] [--csv cache_audit.csv]\n\nReports per month: S2 valid-pixel fraction, S5P valid fraction per pollutant, and\nflags months that are (almost) empty. Also checks the cache is fully finite.\nMemory-mapped: safe on the 1.2 GB S2 array."""\nimport argparse, json, os, numpy as np\n\ndef main(a):\n    meta = json.load(open(os.path.join(a.cache, "meta.json")))\n    ld = lambda n: np.load(os.path.join(a.cache, n + ".npy"), mmap_mode="r")\n    s2, s2ok = ld("s2_month"), ld("s2_month_ok")            # (M,12,H8,W8) fp16, (M,H8,W8)\n    s5ok, s5m = ld("s5p_month_ok"), ld("s5p_mask")           # (M,3,H5,W5), (T,3,H5,W5)\n    M, T = meta["M"], meta["T"]\n    months = [d[:7] for d in meta["month_start"]]\n    wins = np.array([d[:4] for d in meta["win_start"]])\n    pn = ["NO2", "CO", "SO2"][: s5ok.shape[1]]\n\n    rows = []\n    for m in range(M):\n        ok = np.asarray(s2ok[m]) > 0\n        frac = ok.mean()\n        # bands of "valid" pixels should be non-degenerate (all zeros => bad georeferencing / empty export)\n        zero_valid = float((np.asarray(s2[m][0])[ok] == 0).mean()) if ok.any() else np.nan\n        rows.append([months[m], frac, zero_valid] + [float((np.asarray(s5ok[m, c]) > 0).mean()) for c in range(len(pn))])\n\n    print(f"\\n=== S2 / monthly S5P valid-pixel fraction (M={M} months) ===")\n    print("month    S2valid  S2band0==0 " + " ".join(f"{p:>6}" for p in pn))\n    for r in rows:\n        flag = "  <-- LOW" if r[1] < a.min_valid else ""\n        print(f"{r[0]}   {r[1]:6.3f}   {r[2]:8.3f}  " + " ".join(f"{v:6.3f}" for v in r[3:]) + flag)\n\n    s2f = np.array([r[1] for r in rows])\n    print(f"\\nS2 months with valid < {a.min_valid:.0%}: {(s2f < a.min_valid).sum()}/{M}  | "\n          f"valid<50%: {(s2f < 0.5).sum()}  | median {np.median(s2f):.3f}  min {s2f.min():.3f}  max {s2f.max():.3f}")\n    print("S2 months by valid-fraction bucket:",\n          {k: int(((s2f >= lo) & (s2f < hi)).sum()) for k, (lo, hi) in {"<10%": (0, .1), "10-50%": (.1, .5), "50-90%": (.5, .9), ">=90%": (.9, 1.01)}.items()})\n\n    print("\\n=== 5-day S5P window coverage per year (mean valid-pixel fraction over windows) ===")\n    for y in sorted(set(wins)):\n        idx = np.where(wins == y)[0]\n        cov = [float(np.mean([(np.asarray(s5m[i, c]) > 0).mean() for i in idx])) for c in range(len(pn))]\n        empty = [int(sum((np.asarray(s5m[i, c]) > 0).sum() == 0 for i in idx)) for c in range(len(pn))]\n        print(f"{y}: " + "  ".join(f"{p} cov={v:.3f} empty_windows={e}/{len(idx)}" for p, v, e in zip(pn, cov, empty)))\n\n    print("\\n=== Finiteness ===")\n    bad = sum(int((~np.isfinite(np.asarray(s2[m]))).sum()) for m in range(M))\n    print(f"s2_month non-finite values: {bad:,}  ({\'OK\' if bad == 0 else \'FAIL - rebuild cache\'})")\n    wx = os.path.join(a.cache, "weather_5d.npy")\n    if os.path.exists(wx):\n        w = np.load(wx); print(f"weather_5d {w.shape} non-finite: {int((~np.isfinite(w)).sum())}  all-zero windows: {int((np.abs(w).sum(1) == 0).sum())}")\n\n    if a.csv:\n        import csv\n        with open(a.csv, "w", newline="") as f:\n            wr = csv.writer(f); wr.writerow(["month", "s2_valid", "s2_band0_zero_frac"] + [f"{p}_valid" for p in pn]); wr.writerows(rows)\n        print("saved", a.csv)\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser(); ap.add_argument("--cache", required=True)\n    ap.add_argument("--min_valid", type=float, default=0.10); ap.add_argument("--csv", default=None)\n    main(ap.parse_args())\n')
print('Deployed: check_cache.py')
with open('landuse_baseline.py', 'w') as f:
    f.write('#!/usr/bin/env python3\n"""\nStage A (Rule.md): land-use-inferred pollution baseline.\n\nQuestion: can Sentinel-2 surface reflectance ALONE explain the spatial pattern of the\nlong-term S5P column (NO2 by default)?\n\n  Input       : static S2 dry-season composite (median of Jan-Mar months), 12 bands + NDVI/NDBI/MNDWI/NDRE.\n                No timestamps, no weather, no S5P history (Rule.md rules 1-2).\n  Target      : log of the multi-year mean S5P column per S5P cell (supervision only, Rule 4).\n                S5P is a ~3.5x5.5 km tropospheric COLUMN, not ground-level concentration (Rule 5).\n  Validation  : spatial block K-fold with a buffer ring removed from training (no geographic leakage).\n  Baselines   : global mean, geo_knn (smooth geography / "near Chennai = high" shortcut),\n                ridge on land-use index means.  S2 models must beat geo_knn on held-out blocks.\n  Models      : lgbm_s2 (per-cell S2 statistics + neighbourhood context), cnn_s2 (S2 patch),\n                lgbm_s2_geo (diagnostic: does S2 add information on top of geography?).\n\nUsage (Kaggle):\n  python landuse_baseline.py --cache /kaggle/tmp/cache --out /kaggle/working/runs/landuse --poll NO2 --cnn 1\n"""\nimport argparse, json, os, time, warnings\nimport numpy as np\n\nBANDS = ["B1", "B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B9", "B11", "B12"]\nPOLL = {"NO2": 0, "CO": 1, "SO2": 2}\nIDX_NAMES = ["NDVI", "NDBI", "MNDWI", "NDRE"]\n\n\ndef log(*a):\n    print(time.strftime("%H:%M:%S"), *a, flush=True)\n\n\n# ----------------------------------------------------------------------------- data\ndef s2_composite(cache, meta, months):\n    """Per-subpixel median over valid S2 months whose calendar month is in `months`."""\n    s2m = np.load(os.path.join(cache, "s2_month.npy"), mmap_mode="r")\n    s2ok = np.load(os.path.join(cache, "s2_month_ok.npy"), mmap_mode="r")\n    idx = [k for k, d in enumerate(meta["month_start"]) if int(d[5:7]) in months]\n    ok = np.stack([np.asarray(s2ok[k]) > 0 for k in idx])                  # (n, H8, W8)\n    C, H8, W8 = s2m.shape[1:]\n    comp = np.full((C, H8, W8), np.nan, np.float32)\n    for c in range(C):\n        x = np.stack([np.asarray(s2m[k, c], np.float32) for k in idx])\n        x[~ok] = np.nan\n        with warnings.catch_warnings():\n            warnings.simplefilter("ignore", RuntimeWarning)\n            comp[c] = np.nanmedian(x, 0)\n    return comp, ok.sum(0), len(idx)\n\n\ndef spectral_indices(comp):\n    b = lambda n: comp[BANDS.index(n)]\n    nd = lambda a, c: np.clip((a - c) / (a + c + 1e-6), -1, 1)\n    with np.errstate(all="ignore"):\n        return np.stack([nd(b("B8"), b("B4")), nd(b("B11"), b("B8")), nd(b("B3"), b("B11")), nd(b("B8"), b("B5"))]).astype(np.float32)\n\n\ndef s5p_mean(cache, meta, p, years):\n    v = np.load(os.path.join(cache, "s5p_val.npy"), mmap_mode="r")\n    m = np.load(os.path.join(cache, "s5p_mask.npy"), mmap_mode="r")\n    yrs = np.array([int(d[:4]) for d in meta["win_start"]])\n    sel = np.where(np.isin(yrs, list(years)))[0]\n    vv, mm = np.asarray(v[sel, p], np.float64), np.asarray(m[sel, p]) > 0\n    cnt = mm.sum(0)\n    with np.errstate(all="ignore"):\n        mean = np.where(cnt > 0, (vv * mm).sum(0) / np.maximum(cnt, 1), np.nan)\n    return mean, cnt\n\n\n# ----------------------------------------------------------------------------- features\ndef to_cells(a, R, H5, W5):\n    """(K, H8, W8) -> (K, H5, W5, R*R) sub-pixels per S5P cell."""\n    K = a.shape[0]\n    return a[:, :H5 * R, :W5 * R].reshape(K, H5, R, W5, R).transpose(0, 1, 3, 2, 4).reshape(K, H5, W5, R * R)\n\n\ndef box_mean(a, w, r):\n    """Weighted mean over a (2r+1)^2 cell neighbourhood. a: (K,H,W) may contain NaN, w: (H,W)."""\n    K, H, W = a.shape\n    a0 = np.nan_to_num(a) * w\n    pa, pw = np.pad(a0, ((0, 0), (r, r), (r, r))), np.pad(w, r)\n    s, sw = np.zeros_like(a0), np.zeros_like(w)\n    for di in range(2 * r + 1):\n        for dj in range(2 * r + 1):\n            s += pa[:, di:di + H, dj:dj + W]; sw += pw[di:di + H, dj:dj + W]\n    return s / np.maximum(sw, 1e-6)\n\n\ndef build_features(comp, ind, R, H5, W5, ctx_radii=(1, 3)):\n    stack = np.concatenate([comp, ind])                                      # (16, H8, W8), NaN = no data\n    names_ch = BANDS + IDX_NAMES\n    x = to_cells(stack, R, H5, W5)\n    valid_sub = np.isfinite(x[0])\n    land_frac = valid_sub.mean(-1)                                           # (H5, W5)\n    with warnings.catch_warnings():\n        warnings.simplefilter("ignore", RuntimeWarning)\n        mean, std = np.nanmean(x, -1), np.nanstd(x, -1)\n        p10, p90 = np.nanpercentile(x, [10, 90], axis=-1)\n        ndvi, ndbi, mndwi = x[16 - 4], x[16 - 3], x[16 - 2]\n        f = lambda cond: np.where(valid_sub, cond, np.nan)\n        fr = np.stack([np.nanmean(f((ndbi > 0) & (ndvi < 0.2)), -1),         # built-up / bare impervious\n                       np.nanmean(f(ndvi > 0.4), -1),                          # dense vegetation\n                       np.nanmean(f(mndwi > 0), -1)])                          # water\n    fr_names = ["frac_built", "frac_veg", "frac_water"]\n    feats = [mean, std, p10, p90, fr]\n    names = [f"{s}_{n}" for s in ("mean", "std", "p10", "p90") for n in names_ch] + fr_names\n    for r in ctx_radii:                                                      # transport context: neighbouring land use\n        ctx = box_mean(np.concatenate([mean, fr]), land_frac, r)\n        feats.append(ctx); names += [f"ctx{2*r+1}_{n}" for n in names_ch + fr_names]\n    F = np.concatenate(feats).astype(np.float32)                             # (nf, H5, W5)\n    return F, names, land_frac\n\n\n# ----------------------------------------------------------------------------- CV\ndef spatial_folds(H, W, block, k, seed):\n    bid = (np.arange(H)[:, None] // block) * 100000 + (np.arange(W)[None, :] // block)\n    ub = np.unique(bid)\n    perm = np.random.default_rng(seed).permutation(len(ub))\n    lut = {b: perm[i] % k for i, b in enumerate(ub)}\n    return np.vectorize(lut.get)(bid)\n\n\ndef dilate(mask, r):\n    H, W = mask.shape; p = np.pad(mask, r); out = np.zeros_like(mask)\n    for di in range(2 * r + 1):\n        for dj in range(2 * r + 1):\n            out |= p[di:di + H, dj:dj + W]\n    return out\n\n\ndef rank(a):\n    r = np.empty(len(a)); r[np.argsort(a)] = np.arange(len(a)); return r\n\n\ndef metrics(y, p):\n    ok = np.isfinite(p); y, p = y[ok], p[ok]\n    r2 = 1 - ((y - p) ** 2).sum() / ((y - y.mean()) ** 2).sum()\n    top, ptop = y >= np.quantile(y, 0.9), p >= np.quantile(p, 0.9)\n    return dict(R2=r2, RMSE=float(np.sqrt(((y - p) ** 2).mean())), pearson=float(np.corrcoef(y, p)[0, 1]),\n                spearman=float(np.corrcoef(rank(y), rank(p))[0, 1]),\n                hotspot_precision_top10=float((top & ptop).sum() / max(ptop.sum(), 1)), n=int(ok.sum()))\n\n\n# ----------------------------------------------------------------------------- models\ndef fit_predict(name, Xtr, ytr, Xte, seed):\n    if name == "mean":\n        return np.full(len(Xte), ytr.mean())\n    if name == "geo_knn":\n        from sklearn.neighbors import KNeighborsRegressor\n        return KNeighborsRegressor(n_neighbors=10, weights="distance").fit(Xtr, ytr).predict(Xte)\n    if name == "ridge_indices":\n        from sklearn.linear_model import RidgeCV\n        from sklearn.pipeline import make_pipeline\n        from sklearn.preprocessing import StandardScaler\n        return make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 12))).fit(Xtr, ytr).predict(Xte)\n    if name.startswith("lgbm"):\n        import lightgbm as lgb\n        m = lgb.LGBMRegressor(n_estimators=800, learning_rate=0.03, num_leaves=31, min_child_samples=20,\n                              subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=1.0,\n                              random_state=seed, verbose=-1)\n        return m.fit(Xtr, ytr).predict(Xte)\n    raise ValueError(name)\n\n\ndef cnn_oof(comp, ind, land_ok_sub, R, ctx, cells, y, folds_cell, train_sel_fn, epochs, seed):\n    """Small CNN on the raw S2 patch around each cell. Trained per fold, fixed epochs (no test peeking)."""\n    import torch, torch.nn as nn\n    torch.manual_seed(seed); np.random.seed(seed)\n    dev = "cuda" if torch.cuda.is_available() else "cpu"\n    valid = np.isfinite(comp[0])\n    bands = comp.copy()\n    for c in range(bands.shape[0]):                       # robust per-band scaling (inputs only, unsupervised)\n        v = bands[c][valid]; med, iqr = np.median(v), np.subtract(*np.percentile(v, [75, 25])) + 1e-6\n        bands[c] = (bands[c] - med) / iqr\n    X = np.concatenate([np.nan_to_num(np.clip(bands, -5, 5)), np.nan_to_num(ind), valid[None].astype(np.float32)])\n    pad = ctx * R; P = (2 * ctx + 1) * R\n    X = torch.from_numpy(np.pad(X, ((0, 0), (pad, pad), (pad, pad)))).float().to(dev)\n    cin = X.shape[0]\n\n    def patches(cc):\n        return torch.stack([X[:, i * R:i * R + P, j * R:j * R + P] for i, j in cc])\n\n    def blk(a, b): return nn.Sequential(nn.Conv2d(a, b, 3, padding=1), nn.GroupNorm(8, b), nn.GELU())\n\n    oof = np.full(len(y), np.nan)\n    for f in np.unique(folds_cell):\n        tr, te = train_sel_fn(f), np.where(folds_cell == f)[0]\n        mu, sd = y[tr].mean(), y[tr].std() + 1e-8\n        net = nn.Sequential(blk(cin, 32), blk(32, 32), nn.MaxPool2d(2), blk(32, 64), blk(64, 64), nn.MaxPool2d(2),\n                            blk(64, 128), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.2), nn.Linear(128, 1)).to(dev)\n        opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)\n        steps = epochs * int(np.ceil(len(tr) / 128)); sched = torch.optim.lr_scheduler.OneCycleLR(opt, 1e-3, total_steps=steps)\n        yt = torch.tensor((y[tr] - mu) / sd, dtype=torch.float32, device=dev)\n        net.train()\n        for ep in range(epochs):\n            perm = np.random.permutation(len(tr))\n            for s in range(0, len(tr), 128):\n                b = perm[s:s + 128]; xb = patches(cells[tr[b]])\n                if np.random.rand() < 0.5: xb = xb.flip(-1)            # flips only: keep N/S-E/W layout mostly intact\n                if np.random.rand() < 0.5: xb = xb.flip(-2)\n                loss = nn.functional.huber_loss(net(xb).squeeze(1), yt[b], delta=1.0)\n                opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()\n        net.eval(); pr = []\n        with torch.no_grad():\n            for s in range(0, len(te), 256):\n                pr.append(net(patches(cells[te[s:s + 256]])).squeeze(1).cpu().numpy())\n        oof[te] = np.concatenate(pr) * sd + mu\n        log(f"  cnn fold {f}: train {len(tr)} test {len(te)} R2={metrics(y[te], oof[te])[\'R2\']:.3f}")\n    return oof\n\n\n# ----------------------------------------------------------------------------- main\ndef main(a):\n    os.makedirs(a.out, exist_ok=True)\n    meta = json.load(open(os.path.join(a.cache, "meta.json")))\n    H5, W5, R = meta["H5"], meta["W5"], int(meta.get("s2_ratio", meta["H8"] // meta["H5"]))\n    p = POLL[a.poll]; years = list(range(a.y0, a.y1 + 1)); months = [int(m) for m in a.months.split(",")]\n\n    log(f"S2 composite: calendar months {months} ...")\n    comp, nobs, nmon = s2_composite(a.cache, meta, months)\n    ind = spectral_indices(comp)\n    log(f"  used {nmon} monthly composites; sub-pixels with >=1 valid month: {np.isfinite(comp[0]).mean():.3f}")\n\n    tgt, cnt = s5p_mean(a.cache, meta, p, years)\n    F, names, land_frac = build_features(comp, ind, R, H5, W5)\n\n    valid = (cnt >= a.min_obs) & (land_frac >= a.min_land) & np.isfinite(tgt) & (tgt > 0) & np.isfinite(F).all(0)\n    cells = np.argwhere(valid)\n    y = np.log(tgt[valid]) if a.log_target else tgt[valid]\n    X = F[:, valid].T\n    log(f"target {a.poll} mean {years[0]}-{years[-1]}: {len(y)} valid land cells of {H5*W5} "\n        f"(min_obs={a.min_obs}, min_land={a.min_land}); features {X.shape[1]}")\n\n    # reliability ceiling: how reproducible is the target map itself? (split-half by years)\n    h = len(years) // 2\n    t1, _ = s5p_mean(a.cache, meta, p, years[:h]); t2, _ = s5p_mean(a.cache, meta, p, years[h:])\n    ok = valid & np.isfinite(t1) & np.isfinite(t2) & (t1 > 0) & (t2 > 0)\n    f_ = np.log if a.log_target else (lambda z: z)\n    r_half = float(np.corrcoef(f_(t1[ok]), f_(t2[ok]))[0, 1]); rel = 2 * r_half / (1 + r_half)\n    log(f"target reliability: split-half r={r_half:.3f} (years {years[:h]} vs {years[h:]}) -> Spearman-Brown {rel:.3f} "\n        f"(approx. ceiling on explainable variance)")\n\n    geo = cells.astype(np.float32)\n    idx_cols = [i for i, n in enumerate(names) if any(k in n for k in IDX_NAMES + ["frac_"]) and not n.startswith(("std_", "p10_", "p90_"))]\n    designs = {"mean": X[:, :1], "geo_knn": geo, "ridge_indices": X[:, idx_cols], "lgbm_s2": X,\n               "lgbm_s2_geo": np.concatenate([X, geo], 1)}\n\n    rows, oof_all = [], {}\n    for block in [int(b) for b in a.blocks.split(",")]:\n        fold_map = spatial_folds(H5, W5, block, a.k, a.seed)\n        fc = fold_map[valid]\n\n        def train_sel(f, fold_map=fold_map):\n            test = (fold_map == f) & valid\n            excl = dilate(test, a.buffer)\n            return np.where(~excl[valid])[0]\n\n        log(f"--- spatial block {block} cells (~{block*5} km), {a.k} folds, buffer {a.buffer} cells ---")\n        for name, D in designs.items():\n            oof = np.full(len(y), np.nan)\n            for f in range(a.k):\n                tr, te = train_sel(f), np.where(fc == f)[0]\n                if len(te) == 0: continue\n                oof[te] = fit_predict(name, D[tr], y[tr], D[te], a.seed)\n            m = metrics(y, oof); m.update(model=name, block=block); rows.append(m); oof_all[f"{name}_b{block}"] = oof\n            log(f"  {name:14s} R2={m[\'R2\']:.3f} spearman={m[\'spearman\']:.3f} hotspot@10%={m[\'hotspot_precision_top10\']:.2f}")\n        if a.cnn:\n            oof = cnn_oof(comp, ind, None, R, a.ctx, cells, y, fc, train_sel, a.epochs, a.seed)\n            m = metrics(y, oof); m.update(model="cnn_s2", block=block); rows.append(m); oof_all[f"cnn_s2_b{block}"] = oof\n            log(f"  {\'cnn_s2\':14s} R2={m[\'R2\']:.3f} spearman={m[\'spearman\']:.3f} hotspot@10%={m[\'hotspot_precision_top10\']:.2f}")\n\n    import pandas as pd\n    df = pd.DataFrame(rows)[["block", "model", "R2", "RMSE", "pearson", "spearman", "hotspot_precision_top10", "n"]]\n    df.to_csv(os.path.join(a.out, f"landuse_{a.poll}_results.csv"), index=False)\n    np.savez_compressed(os.path.join(a.out, f"landuse_{a.poll}_oof.npz"), cells=cells, y=y, **oof_all)\n    print("\\n" + df.to_string(index=False, float_format=lambda v: f"{v:.3f}"))\n\n    print("\\n=== VERDICT (per block size) ===")\n    for block, g in df.groupby("block"):\n        r = g.set_index("model")["R2"]\n        best_s2 = max((m for m in r.index if m in ("lgbm_s2", "cnn_s2", "ridge_indices")), key=lambda m: r[m])\n        print(f"block {block}: best S2-only = {best_s2} R2 {r[best_s2]:.3f} | geography baseline geo_knn {r[\'geo_knn\']:.3f} "\n              f"| S2 beats geography: {\'YES\' if r[best_s2] > r[\'geo_knn\'] else \'NO\'} "\n              f"| S2 adds over geography (lgbm_s2_geo - geo_knn): {r[\'lgbm_s2_geo\'] - r[\'geo_knn\']:+.3f} "\n              f"| reliability ceiling ~{rel:.3f}")\n\n    # feature importance of the S2-only LightGBM fitted on all cells (descriptive only)\n    import lightgbm as lgb\n    m = lgb.LGBMRegressor(n_estimators=800, learning_rate=0.03, num_leaves=31, min_child_samples=20, subsample=0.8,\n                          subsample_freq=1, colsample_bytree=0.7, random_state=a.seed, verbose=-1, importance_type="gain").fit(X, y)\n    imp = sorted(zip(m.feature_importances_, names), reverse=True)[:15]\n    print("\\nTop S2 features by gain (descriptive, all cells):"); [print(f"  {n:24s} {v:.1f}") for v, n in imp]\n\n    try:\n        import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt\n        blk = int(a.blocks.split(",")[0]); key = f"cnn_s2_b{blk}" if a.cnn else f"lgbm_s2_b{blk}"\n        maps = {}\n        for k, v in (("target", y), ("S2 prediction (" + key + ")", oof_all[key]), ("residual", y - oof_all[key])):\n            M = np.full((H5, W5), np.nan); M[cells[:, 0], cells[:, 1]] = v; maps[k] = M\n        fig, ax = plt.subplots(1, 3, figsize=(15, 6))\n        vmin, vmax = np.nanpercentile(maps["target"], [2, 98])\n        for i, (k, M) in enumerate(maps.items()):\n            im = ax[i].imshow(M, cmap="RdBu_r" if k == "residual" else "viridis",\n                              **({} if k == "residual" else dict(vmin=vmin, vmax=vmax)))\n            ax[i].set_title(k); fig.colorbar(im, ax=ax[i], shrink=0.7)\n        fig.suptitle(f"{a.poll} {years[0]}-{years[-1]} mean column (log={a.log_target}), spatial block {blk}")\n        fig.savefig(os.path.join(a.out, f"landuse_{a.poll}_maps.png"), dpi=110, bbox_inches="tight")\n    except Exception as e:\n        log("map plot skipped:", e)\n    log("done ->", a.out)\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--cache", required=True); ap.add_argument("--out", default="runs/landuse")\n    ap.add_argument("--poll", default="NO2", choices=list(POLL)); ap.add_argument("--y0", type=int, default=2019)\n    ap.add_argument("--y1", type=int, default=2024); ap.add_argument("--months", default="1,2,3")\n    ap.add_argument("--min_obs", type=int, default=40); ap.add_argument("--min_land", type=float, default=0.5)\n    ap.add_argument("--log_target", type=int, default=1); ap.add_argument("--blocks", default="8,4,16")\n    ap.add_argument("--k", type=int, default=5); ap.add_argument("--buffer", type=int, default=2)\n    ap.add_argument("--cnn", type=int, default=1); ap.add_argument("--ctx", type=int, default=2)\n    ap.add_argument("--epochs", type=int, default=40); ap.add_argument("--seed", type=int, default=0)\n    main(ap.parse_args())\n')
print('Deployed: landuse_baseline.py')


In [ ]:
import os, gzip, base64
os.makedirs('weather', exist_ok=True)
wx_b64 = '...'
with open('weather/era5_tn_clusters.csv', 'wb') as f:
    f.write(gzip.decompress(base64.b64decode(wx_b64)))
print('Extracted: weather/era5_tn_clusters.csv (1.6 MB, 5 regional clusters, 2019-2024)')


In [ ]:
import os, shutil, glob

os.makedirs('data2/s2_composites', exist_ok=True)
os.makedirs('data2/s5p_composites', exist_ok=True)

kaggle_input = '/kaggle/input'
found_s2, found_s5p = 0, 0

# 1. Universal recursive discovery for Kaggle Datasets
for root, dirs, files in os.walk(kaggle_input):
    for f in files:
        if f.startswith('s2_') and f.endswith('.tif'):
            dst = os.path.join('data2/s2_composites', f)
            if not os.path.exists(dst): os.symlink(os.path.join(root, f), dst)
            found_s2 += 1
        elif f.startswith('s5p_') and f.endswith('.tif'):
            dst = os.path.join('data2/s5p_composites', f)
            if not os.path.exists(dst): os.symlink(os.path.join(root, f), dst)
            found_s5p += 1

if found_s2 > 0 and found_s5p > 0:
    print(f'Mounted Kaggle Dataset detected! Fast-linked {found_s2} S2 and {found_s5p} S5P GeoTIFF composites (0s, 0 disk used).')
else:
    print('Dataset not mounted in /kaggle/input. Downloading via upgraded gdown...')
    !pip install -q --upgrade gdown
    import gdown
    DRIVE_URL = 'https://drive.google.com/drive/folders/1SQdU-L5SctKcsYXa4WTcto8_emaCurVQ'
    try:
        !gdown "{DRIVE_URL}" --folder -O data2/raw/
    except Exception as e:
        print('Download warning:', e)
    for f in glob.glob('data2/raw/**/s2_*.tif', recursive=True):
        shutil.move(f, os.path.join('data2/s2_composites', os.path.basename(f)))
    for f in glob.glob('data2/raw/**/s5p_*.tif', recursive=True):
        shutil.move(f, os.path.join('data2/s5p_composites', os.path.basename(f)))

s2_count = len(os.listdir('data2/s2_composites'))
s5p_count = len(os.listdir('data2/s5p_composites'))
print(f'Composites Ready for Processing: S2={s2_count}, S5P={s5p_count}')


In [ ]:
import os, glob
print('=== STEP 1: RESOLVING HIGH-RES SATELLITE & WEATHER CACHE ===')

# 1. Check if cache is mounted directly from Kaggle Datasets (Instant & 0 disk usage)
mounted_cache = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 's2_month.npy' in files and 'meta.json' in files:
        mounted_cache = root
        break

if mounted_cache:
    import numpy as _np
    _s2 = _np.load(os.path.join(mounted_cache, 's2_month.npy'), mmap_mode='r')
    _bad = sum(int((~_np.isfinite(_s2[m])).sum()) for m in range(_s2.shape[0]))
    print(f'Mounted cache {mounted_cache}: s2_month {_s2.shape} non-finite values = {_bad:,}')
    if _bad > 0:
        print('STALE CACHE (contains NaN/inf) -> rebuilding with current make_cache.py')
        mounted_cache = None
if mounted_cache:
    print(f'Mounted Kaggle Dataset cache detected at: {mounted_cache}! (Zero disk usage)')
    CACHE_DIR = mounted_cache
else:
    # Use ephemeral /kaggle/tmp (~60 GB available, keeping /kaggle/working 100% clean)
    CACHE_DIR = '/kaggle/tmp/cache' if os.path.exists('/kaggle/tmp') else '/tmp/cache'
    os.makedirs(CACHE_DIR, exist_ok=True)
    print('Downloading verified native R=8 cache from https://www.kaggle.com/datasets/adarshpradeep17/airq-dualclock-cache (~15s)...')
    pass  # skip re-downloading the published cache (it is the stale one)
    if os.path.exists(os.path.join(CACHE_DIR, 'meta.json')):
        print(f'SUCCESS: Cache downloaded and unzipped into {CACHE_DIR}!')
    else:
        print('Building aligned cache in temporary storage (RATIO=8, ~6-8 mins)...')
        !python make_cache.py --s2_dir data2/s2_composites --s5p_dir data2/s5p_composites --out {CACHE_DIR} --s2_ratio 8 --weather weather/era5_tn_clusters.csv --start 2019-01-01 --end 2024-12-31 --zero_is_invalid 1
    print('Cache ready for multi-GPU training!')


In [ ]:
import os, json
print('=== STEP 1B: PERSIST CACHE AS PERMANENT KAGGLE DATASET (ONCE AND FOR ALL) ===')

# Cache is already live at https://www.kaggle.com/datasets/adarshpradeep17/airq-dualclock-cache
PUBLISH_TO_OWN_ACCOUNT = False
if PUBLISH_TO_OWN_ACCOUNT and not mounted_cache and os.path.exists(os.path.join(CACHE_DIR, 'meta.json')):
    print('Setting up Kaggle credentials for antonioroger2...')
    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    # Credentials: add KAGGLE_USERNAME / KAGGLE_KEY as Kaggle Secrets; never hardcode them.
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient(); os.environ['KAGGLE_USERNAME'] = _sec.get_secret('KAGGLE_USERNAME'); os.environ['KAGGLE_KEY'] = _sec.get_secret('KAGGLE_KEY')

    metadata = {
        'title': 'AirQ Dual-Clock High-Res Cache',
        'id': 'antonioroger2/airq-dualclock-highres-cache',
        'licenses': [{'name': 'CC0-1.0'}]
    }
    with open(os.path.join(CACHE_DIR, 'dataset-metadata.json'), 'w') as f:
        json.dump(metadata, f, indent=2)

    print('Uploading cache as permanent Kaggle Dataset (antonioroger2/airq-dualclock-highres-cache)...')
    !kaggle datasets create -p {CACHE_DIR} --dir-mode zip || kaggle datasets version -p {CACHE_DIR} -m 'Native 1:1 high-fidelity cache (R=8, 600m)' --dir-mode zip
    print('\nSUCCESS: Cache published permanently! You can now attach antonioroger2/airq-dualclock-highres-cache to any notebook for instant 0-second mount.')
else:
    print('Cache already mounted from Kaggle Datasets or previously published. Skipping upload.')


In [ ]:
# MODE: 'landuse' = Rule.md Stage A (S2-only -> S5P mean column, spatial block CV); 'dcnet' = temporal DC-Net forecaster
MODE = 'landuse'
print('=== CACHE AUDIT (valid-pixel coverage, finiteness) ===')
!python check_cache.py --cache {CACHE_DIR} --csv /kaggle/working/cache_audit.csv
if MODE == 'landuse':
    print('=== STAGE A: LAND-USE-INFERRED POLLUTION (S2 only, no time/weather/S5P inputs) ===')
    !python -u landuse_baseline.py --cache {CACHE_DIR} --out /kaggle/working/runs/landuse --poll NO2 --cnn 1 --blocks 8,4,16
EPOCHS = 3  # DC-Net smoke test. Set to 120 for the full run.
POLL = 'all'  # 'all' = NO2+CO+SO2 joint model, 'NO2' = single-pollutant baseline
OUT = f'/kaggle/working/runs/dcnet_{POLL}'
if MODE == 'dcnet':
    print(f'=== STEP 2: TRAINING DC-NET (STAGE 1, POLL={POLL}, {EPOCHS} epochs, single GPU) ===')
    !python -u train.py --cache {CACHE_DIR} --out {OUT} --stage 1 --poll {POLL} --long_mem_dropout 0.50 --batch 8 --epochs {EPOCHS} --workers 2 --vram_cap 0.90 --no_dp


In [ ]:
if MODE == 'dcnet':
    print('=== STEP 3: RUNNING EMPIRICAL CLIMATOLOGY EVALUATION (2024 VAL SPLIT) ===')
    !python evaluate.py --ckpt {OUT}/best.pt --cache {CACHE_DIR} --split val --out {OUT}


In [ ]:
import pandas as pd
metrics_path = f'{OUT}/dcnet_val_metrics.csv'
if os.path.exists(metrics_path):
    df = pd.read_csv(metrics_path)
    print('\n' + '='*80)
    print('FINAL EVALUATION METRICS (2024 HELD-OUT SPLIT):')
    print('='*80)
    print(df[['pollutant', 'horizon_days', 'R2', 'pearson_r', 'skill_vs_clim_pct', 'coverage_80', 'csi_p90']].to_string())
else:
    print('Metrics file not found.')


In [ ]:
LAUNCH_APP = False  # the Gradio server never exits, so a batch (push) run would hang until the Kaggle time limit
if LAUNCH_APP:
    print('=== STEP 4: LAUNCHING INTERACTIVE GRADIO DASHBOARD SERVICE ===')
    !python app.py --ckpt {OUT}/best.pt --cache {CACHE_DIR} --share
else:
    print('Skipping dashboard (LAUNCH_APP=False). Run training/eval outputs are in', OUT)
